In [1]:
import pandas as pd
import re
from IPython.display import display
from pathlib import Path
from collections import defaultdict

In [2]:
pop = pd.read_csv(r"C:\Users\MARE\OneDrive\Documents\data311_project2\data\raw\population_addresses_messy.csv")
pop.head()

,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
0,POP_S_7315039,student,7917 J St,Eureka,CA,95501,6863 Cedar Ln,Redding,CA,96001,NorCal
1,POP_S_4194026,Student,1845 Bates Rd,McKinleyville,ca,95519,8942 Birch Pl,La Mesa,CA,91942,SanDiego
2,POP_S_4478952,Student,9921 Harris St,EUREKA,CA,95501,9527 Maple Dr,Redding,CA,96001,NorCal
3,POP_S_2541649,Student,3397 5th St,Eureka,CA,95501,1318 Meadow Blvd,National City,CA,91950,SanDiego
4,POP_S_4662231,Student,8466 H St,Eureka,CA,95501,3406 Meadow Pl,National City,CA,91950,SanDiego


In [3]:
summary = pd.DataFrame({
    "unique_values": pop.nunique(),
    "total_rows": len(pop),
    "all_values_unique": pop.apply(lambda col: col.is_unique),
    "missing_values": pop.isna().sum()
})

print(summary)

                  unique_values  total_rows  all_values_unique  missing_values
pop_id                     7629        7785              False               0
role                         13        7785              False               0
current_street             7449        7785              False              65
current_city                 25        7785              False              66
current_state                 6        7785              False               0
current_zip                  33        7785              False              86
permanent_street           7117        7785              False              56
permanent_city               76        7785              False               0
permanent_state              10        7785              False               0
permanent_zip                38        7785              False               0
permanent_region              6        7785              False               0


In [4]:
# Data Auddit
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 100)

In [5]:
#pop id
print("=" * 80)
print("1. DUPLICATE pop_id VALUES")
print("=" * 80)

duplicate_ids = pop[
    pop["pop_id"].notna()
    & pop["pop_id"].duplicated(keep=False)
].sort_values("pop_id")

if duplicate_ids.empty:
    print("No duplicate pop_id values found.")
else:
    print("Repeated IDs and their occurrence counts:")
    display(
        duplicate_ids["pop_id"]
        .value_counts()
        .rename_axis("pop_id")
        .reset_index(name="occurrences")
    )

    print("\nAll rows associated with repeated IDs:")
    display(duplicate_ids)


1. DUPLICATE pop_id VALUES
Repeated IDs and their occurrence counts:


,pop_id,occurrences
0,POP_S_1371371,3
1,POP_S_4940241,3
2,POP_F_1451911,2
3,POP_F_2001167,2
4,POP_F_2930051,2
5,POP_F_4417903,2
6,POP_F_7389365,2
7,POP_F_7849184,2
8,POP_F_8236540,2
9,POP_F_8683616,2



All rows associated with repeated IDs:


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
6160,POP_F_1451911,Faculty,1100 G St,Arcata,CA,95521,1100 G St,ARCATA,CA,95521,Local
7710,POP_F_1451911,Faculty,1100 G St,Arcata,CA,95521,1100 G St,ARCATA,CA,95521,Local
6305,POP_F_2001167,Faculty,7238 Greenwood Ave,Blue Lake,CA,95525,7238 Greenwood Ave,Blue Lake,CA,95525,Local
7654,POP_F_2001167,Faculty,7238 Greenwood Ave,Blue Lake,CA,95525,7238 Greenwood Ave,Blue Lake,CA,95525,Local
6347,POP_F_2930051,Faculty,366 Q St,Arcata,CA,95521,366 Q St,Arcata,CA,95521,Local
...,...,...,...,...,...,...,...,...,...,...,...
7765,POP_S_9948521,Student,6124 Central Ave,McKinleyville,CA,95519,8447 Vista St,Phoenix,AZ,85004,OutOfState
1896,POP_S_9952831,Student,9825 Union St,Arcata,CA,95521,7419 Pine St,San Jose,CA,95112,BayArea
7664,POP_S_9952831,Student,9825 Union St,Arcata,CA,95521,7419 Pine St,San Jose,CA,95112,BayArea
4879,POP_S_9973271,Student,5488 h st,Arcata,CA,95521,5485 Canyon Dr,Chico,CA,95926,NorCal


In [6]:
#drop duplicate rows
pop_unique = pop.drop_duplicates()
#non-duplicate rows where pop_id is not unique (how do we handle?)
pop_unique[pop_unique.duplicated(subset = "pop_id", keep = False)].sort_values("pop_id")

,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
1427,POP_S_1371371,Student,5576 G St,Arcata,CA,95521,282 Hill Ln,Glendale,CA,91203,LA
7163,POP_S_1371371,Staff/Admin,2570 Myrtle Ave,Eureka,CA,95501,2570 Myrtle Ave,EUREKA,CA,95501,Local
380,POP_S_4940241,Student,8640 Janis Ct,Arcata,OR,95521,7386 Lake Rd,Chico,CA,95926,NorCal
4552,POP_S_4940241,STUDENT,1466 Spears Rd,Arcata,CA,95521,6155 Spruce Dr,San Diego,CA,92103,SanDiego
3880,POP_S_6159896,Student,1695 G St,Arcata,CA,95521,2144 Willow Way,Austin,TX,78701,OutOfState
7097,POP_S_6159896,Staff/Admin,8317 WINCHESTER AVE,McKinleyville,CA,95519,5317 Winchester Ave,McKinleyville,CA,95519,Local
43,POP_S_7366358,Student,2208 Baldwin St,NaN,CA,95521,9805 Meadow Blvd,Fremont,CA,94536,BayArea
3554,POP_S_7366358,Student,2551 Union St,Arcata,CA,95521,421 Lake Rd,Santa Rosa,CA,95401,NorCal


In [7]:
#check that all remaining pop_id are of the correct format
pop_unique[pop_unique["pop_id"].astype(str).str.fullmatch(r"POP_[SFA]_\d+") == False]


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region


In [8]:
# Role audit
print("\n" + "=" * 80)
print("2. ROLE AUDIT")
print("=" * 80)

print("All role values and their frequencies:")
role_counts = (
    pop["role"]
    .value_counts(dropna=False)
    .rename_axis("role")
    .reset_index(name="count")
)
display(role_counts)

# Normalize capitalization and surrounding whitespace
role_normalized = pop["role"].astype("string").str.strip().str.casefold()

print("\nRoles after ignoring capitalization and outer whitespace:")
display(
    pop.assign(normalized_role=role_normalized)
       .groupby("normalized_role", dropna=False)
       .agg(
           count=("role", "size"),
           original_values=("role",
                            lambda x: sorted(x.dropna().unique().tolist()))
       )
       .reset_index()
)



2. ROLE AUDIT
All role values and their frequencies:


,role,count
0,Student,6164
1,Staff/Admin,683
2,Faculty,498
3,STUDENT,137
4,student,115
5,Studnet,112
6,Staff,15
7,faculty,11
8,Faculty,10
9,FACULTY,10



Roles after ignoring capitalization and outer whitespace:


,normalized_role,count,original_values
0,faculty,529,"[FACULTY, Faculty, Faculty , faculty]"
1,staff,15,[Staff]
2,staff-admin,10,[Staff-Admin]
3,staff/admin,703,"[STAFF/ADMIN, Staff/Admin, staff/admin]"
4,student,6416,"[STUDENT, Student, student]"
5,studnet,112,[Studnet]


In [9]:
#standardize role names and confirm 
pop_unique.loc[pop_unique["role"].str.startswith(("stu", "STU", "Stu")), "role"] = "student"
pop_unique.loc[pop_unique["role"].str.startswith(("sta", "STA", "Sta")), "role"] = "staff/admin"
pop_unique.loc[pop_unique["role"].str.startswith(("fac", "FAC", "Fac")), "role"] = "faculty"
pop = pop_unique
pop.query("role not in ['student', 'staff/admin', 'faculty']")


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region


In [10]:
print("All role values and their frequencies:")
role_counts = (
    pop["role"]
    .value_counts(dropna=False)
    .rename_axis("role")
    .reset_index(name="count")
)
display(role_counts)

All role values and their frequencies:


,role,count
0,student,6400
1,staff/admin,713
2,faculty,520


In [11]:
# Current Address Audit
current_columns = [
    "current_street",
    "current_city",
    "current_state",
    "current_zip"
]

print("\n" + "=" * 80)
print("3. CURRENT ADDRESS AUDIT")
print("=" * 80)



3. CURRENT ADDRESS AUDIT


In [12]:
# Current Address Audit
permanent_columns = [
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

print("\n" + "=" * 80)
print("4. PERMANENT ADDRESS AUDIT")
print("=" * 80)



4. PERMANENT ADDRESS AUDIT


In [13]:
def audit_duplicates(df, columns, address_type):
    for column in columns:
        print("\n" + "-" * 70)
        print(f"{address_type}: {column}")
        print("-" * 70)

        # Ignore missing values when identifying repeats
        counts = df[column].value_counts(dropna=True)
        repeated_values = counts[counts > 1]

        if repeated_values.empty:
            print("No repeated non-missing values found.")
            continue

        print("Repeated values and occurrence counts:")
        display(
            repeated_values
            .rename_axis(column)
            .reset_index(name="occurrences")
        )

        # Show every full row associated with a repeated value
        mask = (
            df[column].notna()
            & df[column].isin(repeated_values.index)
        )

        print("All rows containing those repeated values:")
        display(df.loc[mask].sort_values(column))


# Run the current address checks
audit_duplicates(pop, current_columns, "CURRENT ADDRESS")

# Run the permanent address checks
audit_duplicates(pop, permanent_columns, "PERMANENT ADDRESS")


----------------------------------------------------------------------
CURRENT ADDRESS: current_street
----------------------------------------------------------------------
Repeated values and occurrence counts:


,current_street,occurrences
0,-,20
1,unknown,18
2,8571 Q St,3
3,7550 Samoa Blvd,2
4,173 Baldwin St,2
5,8223 K St,2
6,8332 LK Wood Blvd,2
7,3454 Q St,2
8,1457 Samoa Blvd,2
9,2395 Samoa Blvd,2


All rows containing those repeated values:


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
393,POP_S_5042253,student,-,Eureka,CA,95501,9921 Willow Blvd,Daly City,CA,94015,BayArea
764,POP_S_5029972,student,-,Arcata,CA,95521,2446 Meadow Rd,Santa Monica,CA,90401,LA
788,POP_S_9555121,student,-,Eureka,CA,95501,3577 Pine Way,San Jose,CA,95112,BayArea
1377,POP_S_4268441,student,-,Eureka,CA,95501,5725 Ridge Ave,Boulder,CO,80302,OutOfState
1612,POP_S_4312891,student,-,Eureka,CA,95501,2316 Lake Blvd,San Francisco,CA,94110,BayArea
...,...,...,...,...,...,...,...,...,...,...,...
4402,POP_S_2284475,student,unknown,Arcata,CA,95521,7381 Elm Rd,San Francisco,CA,94110,BayArea
5139,POP_S_5937004,student,unknown,Eureka,CA,95501,685 Vista Blvd,Anaheim,CA,92801,LA
5625,POP_S_7304226,student,unknown,Arcata,CA,95521,8817 Canyon Ave,San Diego,CA,92103,SanDiego
5721,POP_S_8041398,student,unknown,Arcata,CA,95521,9281 Meadow Ln,Glendale,CA,91203,LA



----------------------------------------------------------------------
CURRENT ADDRESS: current_city
----------------------------------------------------------------------
Repeated values and occurrence counts:


,current_city,occurrences
0,Arcata,3705
1,Eureka,1609
2,McKinleyville,1223
3,Blue Lake,215
4,Trinidad,172
5,Fortuna,166
6,arcata,66
7,ARCATRA,63
8,ARCATA,58
9,Arcatra,51


... (26 outputs left out)


----------------------------------------------------------------------
PERMANENT ADDRESS: permanent_region
----------------------------------------------------------------------
Repeated values and occurrence counts:


,permanent_region,occurrences
0,LA,1602
1,BayArea,1464
2,OutOfState,1425
3,Local,1233
4,NorCal,1124
5,SanDiego,785


All rows containing those repeated values:


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
17,POP_S_6830775,student,6600 Alliance Rd,Arcata,CA,95521.0,9799 Meadow Pl,Berkeley,CA,94704,BayArea
19,POP_S_6877850,student,7456 Baldwin St,Arcata,CA,95521,5309 Birch Dr,San Jose,CA,95112,BayArea
20,POP_S_6413414,student,5161 LK Wood Blvd,Arcata,CA,95521,8852 Maple Ln,Oakland,CA,94612,BayArea
22,POP_S_8903984,student,7959 Murray Rd,McKinleyville,CA,95519,8231 Lake St,Berkeley,CA,94704,BayArea
26,POP_S_5138664,student,2260 LK Wood Blvd,Arcata,CA,95521,2385 Ridge Pl,Oakland,CA,94612,BayArea
...,...,...,...,...,...,...,...,...,...,...,...
7602,POP_S_56063085,student,2905 St Louis Rd #501,Arcata,CA,95521,3605 Willow Way,Escondido,CA,92025,SanDiego
7623,POP_S_86622944,student,2905 St Louis Rd #311,Arcata,CA,95521,1496 Maple Way,Escondido,CA,92025,SanDiego
7628,POP_S_70346125,student,2905 St Louis Rd #708,ARCATRA,CA,95521,5102 Elm Blvd,National City,CA,91950,SanDiego
7629,POP_S_83353179,student,2905 St Louis Rd #258,Arcata,CA,95521,6387 Canyon Blvd,San Diego,CA,92103,SanDiego


In [14]:
# Find Case and Whitespace
def audit_format_variations(df, columns, address_type):
    for column in columns:
        # Convert to strings, trim outer spaces, ignore case
        normalized = (
            df[column]
            .astype("string")
            .str.strip()
            .str.casefold()
        )

        # Preserve original spellings for comparison
        temp = pd.DataFrame({
            "normalized_value": normalized,
            "original_value": df[column]
        }).dropna(subset=["normalized_value"])

        variations = (
            temp.groupby("normalized_value")["original_value"]
            .agg(lambda x: sorted(x.unique().tolist()))
        )

        # Keep only groups with multiple original representations
        variations = variations[variations.map(len) > 1]

        if not variations.empty:
            print(
                f"\n{address_type} FORMAT VARIATIONS: {column}"
            )
            display(variations.rename("original_variations").reset_index())


print("\n" + "=" * 80)
print("5. CASE AND WHITESPACE VARIATIONS")
print("=" * 80)

audit_format_variations(
    pop,
    ["role"] + current_columns + permanent_columns,
    "ROLE / ADDRESS"
)

print("\n" + "=" * 80)
print("DATA-QUALITY AUDIT COMPLETE")
print("=" * 80)


5. CASE AND WHITESPACE VARIATIONS

ROLE / ADDRESS FORMAT VARIATIONS: current_street


,normalized_value,original_variations
0,3200 janis ct,"[3200 Janis Ct, 3200 janis ct]"
1,322 h st,"[322 H St, 322 h st]"
2,3334 murray rd,"[3334 MURRAY RD, 3334 Murray Rd]"
3,4742 alliance rd,"[4742 ALLIANCE RD, 4742 Alliance Rd]"
4,5073 central ave,"[5073 Central Ave, 5073 central ave]"
5,6141 11th st,"[6141 11th St, 6141 11th st]"
6,6325 myrtle ave,"[6325 Myrtle Ave, 6325 myrtle ave]"
7,636 g st,"[636 G St, 636 g st]"
8,6453 wabash ave,"[6453 Wabash Ave, 6453 wabash ave]"
9,659 h st,"[659 H St, 659 h st]"



ROLE / ADDRESS FORMAT VARIATIONS: current_city


,normalized_value,original_variations
0,arcata,"[ARCATA, Arcata, arcata]"
1,arcatra,"[ARCATRA, Arcatra]"
2,blue lake,"[BLUE LAKE, Blue Lake]"
3,eureka,"[EUREKA, Eureka, eureka]"
4,fortuna,"[Fortuna, fortuna]"
5,mckinleyville,"[MCKINLEYVILLE, McKinleyville, mckinleyville]"
6,trinidad,"[TRINIDAD, Trinidad]"



ROLE / ADDRESS FORMAT VARIATIONS: current_state


,normalized_value,original_variations
0,ca,"[CA, Ca, ca]"



ROLE / ADDRESS FORMAT VARIATIONS: permanent_city


,normalized_value,original_variations
0,anaheim,"[ANAHEIM, Anaheim]"
1,arcata,"[ARCATA, Arcata]"
2,austin,"[AUSTIN, Austin]"
3,berkeley,"[BERKELEY, Berkeley]"
4,blue lake,"[BLUE LAKE, Blue Lake]"
5,boise,"[BOISE, Boise]"
6,boulder,"[BOULDER, Boulder]"
7,brooklyn,"[BROOKLYN, Brooklyn]"
8,chico,"[CHICO, Chico]"
9,chula vista,"[CHULA VISTA, Chula Vista]"



DATA-QUALITY AUDIT COMPLETE


# Clean address

In [15]:
print("=" * 80)
print("ADDRESS DATA CLEANING")
print("=" * 80)

# Define address columns
current_columns = [
    "current_street",
    "current_city",
    "current_state",
    "current_zip"
]

permanent_columns = [
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

address_columns = current_columns + permanent_columns


# just in case copy data to preserve previous role work.

# Create a snapshot of the current working DataFrame.
# This includes the role-cleaning changes made previously.

pop_before_address_cleaning = pop.copy(deep=True)

print("Records before address cleaning:", len(pop))
print("Address columns being processed:", len(address_columns))


# whitespace cleanup
# Remove leading and trailing whitespace from address fields.
# Preserve missing values rather than converting them to strings.

for column in address_columns:

    pop[column] = (
        pop[column]
        .astype("string")
        .str.strip()
    )


# standardize city name
# Explicit corrections for known variations.
# Keys must be lowercase, stripped representations.

city_mapping = {

    # Arcata
    "arcata": "Arcata",
    "arcatra": "Arcata",

    # Eureka
    "eureka": "Eureka",
    "eureaka": "Eureka",
    "eurika": "Eureka",

    # McKinleyville
    "mckinleyville": "McKinleyville",
    "mckinelyville": "McKinleyville",

    # Blue Lake
    "blue lake": "Blue Lake",
    "bluelake": "Blue Lake",

    # Trinidad
    "trinidad": "Trinidad",
    "trinadad": "Trinidad",

    # Fortuna
    "fortuna": "Fortuna",
    "fortunna": "Fortuna",

    # Other cities observed in the permanent address audit
    "glendale": "Glendale",
    "anaheim": "Anaheim",
    "fremont": "Fremont",
    "los angeles": "Los Angeles",
    "pasadena": "Pasadena",
    "santa monica": "Santa Monica",
    "daly city": "Daly City",
    "long beach": "Long Beach",
    "berkeley": "Berkeley",
    "san francisco": "San Francisco",
    "oakland": "Oakland",
    "san jose": "San Jose",
    "redding": "Redding",
    "ukiah": "Ukiah",
    "sacramento": "Sacramento",
    "santa rosa": "Santa Rosa",
    "chico": "Chico",
    "boulder": "Boulder",
    "salt lake city": "Salt Lake City",
    "portland": "Portland",
    "seattle": "Seattle",
    "austin": "Austin",
    "national city": "National City",
    "brooklyn": "Brooklyn",
    "phoenix": "Phoenix",
    "reno": "Reno",
    "boise": "Boise",
    "la mesa": "La Mesa",
    "oceanside": "Oceanside",
    "escondido": "Escondido",
    "chula vista": "Chula Vista",
    "san diego": "San Diego"
}


#Apply standardization
city_columns = [
    "current_city",
    "permanent_city"
]

for column in city_columns:

    normalized_city = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    # Convert known missing-value placeholders to actual missing values.
    normalized_city = normalized_city.mask(
        normalized_city.isin(["", "-", "unknown", "n/a", "na"])
    )

    # Correct known city variations.
    # Unrecognized city names are retained for manual review.
    pop[column] = normalized_city.map(city_mapping).where(
        normalized_city.isin(city_mapping),
        pop[column]
    )

    # Standardize recognized missing-value placeholders.
    pop[column] = pop[column].mask(
        normalized_city.isin(["", "-", "unknown", "n/a", "na"])
    )


# Standard State
state_columns = [
    "current_state",
    "permanent_state"
]

state_mapping = {
    "ca": "CA",
    "california": "CA",
    "or": "OR",
    "oregon": "OR"
}

for column in state_columns:

    normalized_state = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.casefold()
        .str.replace(r"\.$", "", regex=True)
    )

    normalized_state = normalized_state.mask(
        normalized_state.isin(["", "-", "unknown", "n/a", "na"])
    )

    # Standardize recognized state names and abbreviations.
    # Preserve other nonmissing state values for review.
    pop[column] = normalized_state.map(state_mapping).where(
        normalized_state.isin(state_mapping),
        pop[column]
    )

    pop[column] = pop[column].mask(
        normalized_state.isin(["", "-", "unknown", "n/a", "na"])
    )

# update current_state based on current_city

# Cities that should have California as their current state.
california_cities = [
    "Arcata",
    "Eureka",
    "McKinleyville",
    "Blue Lake",
    "Trinidad",
    "Fortuna"
]

# Normalize city values for case-insensitive matching.
normalized_city = (
    pop["current_city"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Create a case-insensitive list of target cities.
target_cities = [city.casefold() for city in california_cities]

# Identify matching records.
mask = normalized_city.isin(target_cities)

# Count records that will be updated.
records_to_update = mask.sum()

# Update current_state to CA.
pop.loc[mask, "current_state"] = "CA"

print(f"Records updated: {records_to_update:,}")
print("Current state successfully standardized to CA.")

# standard street format
street_columns = [
    "current_street",
    "permanent_street"
]

for column in street_columns:

    pop[column] = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Convert blank strings and explicit placeholders to missing.
    normalized_street = pop[column].str.casefold()

    pop[column] = pop[column].mask(
        normalized_street.isin(["", "-", "unknown", "n/a", "na"])
    )

# standardize city and zip

# Define the city-to-ZIP mapping.
city_zip_mapping = {
    "Arcata": "95521",
    "Eureka": "95501",
    "McKinleyville": "95519",
    "Blue Lake": "95525",
    "Trinidad": "95570",
    "Fortuna": "95540"
}

# Create the reverse ZIP-to-city mapping.
zip_city_mapping = {
    zip_code: city
    for city, zip_code in city_zip_mapping.items()
}

# normalize city and zip

normalized_city = (
    pop["current_city"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

normalized_zip = (
    pop["current_zip"]
    .astype("string")
    .str.strip()
)

# update current_zip based on current_city

city_lookup = {
    city.casefold(): zip_code
    for city, zip_code in city_zip_mapping.items()
}

city_mask = normalized_city.isin(city_lookup)

pop.loc[city_mask, "current_zip"] = (
    normalized_city.loc[city_mask].map(city_lookup)
)

# update curent_city based on current_zip

zip_mask = normalized_zip.isin(zip_city_mapping)

pop.loc[zip_mask, "current_city"] = (
    normalized_zip.loc[zip_mask].map(zip_city_mapping)
)

# display

print("=" * 70)
print("CURRENT CITY AND ZIP STANDARDIZATION")
print("=" * 70)

print(f"Records with city-based ZIP updates: {city_mask.sum():,}")
print(f"Records with ZIP-based city updates: {zip_mask.sum():,}")

print("\nUpdated city and ZIP combinations:")

verification = (
    pop.loc[
        pop["current_city"].isin(city_zip_mapping.keys())
        | pop["current_zip"].isin(zip_city_mapping.keys()),
        ["current_city", "current_zip"]
    ]
    .value_counts()
    .reset_index(name="records")
    .sort_values(["current_city", "current_zip"])
)

display(verification)

#standard zip
zip_columns = [
    "current_zip",
    "permanent_zip"
]

for column in zip_columns:

    # Keep ZIP codes as strings to preserve leading zeros.
    zip_values = (
        pop[column]
        .astype("string")
        .str.strip()
    )

    normalized_zip = zip_values.str.casefold()

    # Convert missing-value placeholders to actual missing values.
    zip_values = zip_values.mask(
        normalized_zip.isin(["", "-", "unknown", "n/a", "na"])
    )

    # Remove a trailing .0 only when the entire value is numeric
    # followed by a decimal point and zero.
    zip_values = zip_values.str.replace(
        r"^(\d+)\.0$",
        r"\1",
        regex=True
    )

    # Pad numeric ZIP codes of one to four digits with leading zeros.
    # Preserve five-digit ZIP codes and ZIP+4 values.
    valid_short_zip = zip_values.str.fullmatch(r"\d{1,4}", na=False)

    zip_values.loc[valid_short_zip] = (
        zip_values.loc[valid_short_zip].str.zfill(5)
    )

    pop[column] = zip_values


# result output
print("\n" + "=" * 80)
print("ADDRESS CLEANING SUMMARY")
print("=" * 80)

print("Records after address cleaning:", len(pop))

print("\nMissing values by address column:")

display(
    pop[address_columns]
    .isna()
    .sum()
    .rename("missing_values")
    .to_frame()
)

print("\nUnique values by address column:")

display(
    pop[address_columns]
    .nunique()
    .rename("unique_values")
    .to_frame()
)

print("\nAddress cleaning completed.")
print("The cleaned values currently exist in the working DataFrame.")
print("The CSV has not yet been overwritten or exported.")

ADDRESS DATA CLEANING
Records before address cleaning: 7633
Address columns being processed: 9
Records updated: 7,527
Current state successfully standardized to CA.
CURRENT CITY AND ZIP STANDARDIZATION
Records with city-based ZIP updates: 7,527
Records with ZIP-based city updates: 7,212

Updated city and ZIP combinations:


,current_city,current_zip,records
0,Arcata,95521,3993
3,Blue Lake,95525,231
1,Eureka,95501,1744
5,Fortuna,95540,178
2,McKinleyville,95519,1303
4,Trinidad,95570,184



ADDRESS CLEANING SUMMARY
Records after address cleaning: 7633

Missing values by address column:


,missing_values
current_street,101
current_city,0
current_state,0
current_zip,0
permanent_street,111
permanent_city,0
permanent_state,0
permanent_zip,0
permanent_region,0



Unique values by address column:


,unique_values
current_street,7447
current_city,6
current_state,1
current_zip,6
permanent_street,7115
permanent_city,38
permanent_state,10
permanent_zip,38
permanent_region,6



Address cleaning completed.
The cleaned values currently exist in the working DataFrame.
The CSV has not yet been overwritten or exported.


In [16]:
# re audit
print("=" * 80)
print("CURRENT ADDRESS AUDIT AFTER CLEANING")
print("=" * 80)

audit_duplicates(
    pop,
    current_columns,
    "CURRENT ADDRESS AFTER CLEANING"
)

print("\n" + "=" * 80)
print("PERMANENT ADDRESS AUDIT AFTER CLEANING")
print("=" * 80)

audit_duplicates(
    pop,
    permanent_columns,
    "PERMANENT ADDRESS AFTER CLEANING"
)

CURRENT ADDRESS AUDIT AFTER CLEANING

----------------------------------------------------------------------
CURRENT ADDRESS AFTER CLEANING: current_street
----------------------------------------------------------------------
Repeated values and occurrence counts:


,current_street,occurrences
0,8571 Q St,3
1,7550 Samoa Blvd,2
2,173 Baldwin St,2
3,8223 K St,2
4,8332 LK Wood Blvd,2
5,3454 Q St,2
6,1457 Samoa Blvd,2
7,2395 Samoa Blvd,2
8,1021 5th St,2
9,3228 Hiller Rd,2


All rows containing those repeated values:


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
1489,POP_S_6911832,student,1009 Main St,Fortuna,CA,95540,3329 Willow Blvd,Chico,CA,95926,NorCal
3705,POP_S_1072070,student,1009 Main St,Trinidad,CA,95570,9329 Birch Blvd,San Diego,CA,92103,SanDiego
338,POP_S_7547868,student,1021 5th St,Eureka,CA,95501,1906 Ridge Ln,National City,CA,91950,SanDiego
4917,POP_S_7729000,student,1021 5th St,Eureka,CA,95501,7113 Hill St,Boulder,CO,80302,OutOfState
874,POP_S_5764958,student,1087 Hiller Rd,McKinleyville,CA,95519,3376 Oak Way,Redding,CA,96001,NorCal
1752,POP_S_2084657,student,1087 Hiller Rd,McKinleyville,CA,95519,6146 Oak Rd,Oakland,CA,94612,BayArea
5584,POP_S_3378679,student,1120 11th St,Arcata,CA,95521,9919 Ridge Way,Santa Monica,CA,90401,LA
5624,POP_S_9217676,student,1120 11th St,Arcata,CA,95521,1256 Willow Ln,Redding,CA,96001,NorCal
1757,POP_S_6143981,student,1286 Baldwin St,Arcata,CA,95521,608 Sunset St,Brooklyn,NY,11211,OutOfState
6242,POP_F_6104810,faculty,1286 Baldwin St,Arcata,CA,95521,1286 Baldwin St,Arcata,CA,95521,Local



----------------------------------------------------------------------
CURRENT ADDRESS AFTER CLEANING: current_city
----------------------------------------------------------------------
Repeated values and occurrence counts:


,current_city,occurrences
0,Arcata,3993
1,Eureka,1744
2,McKinleyville,1303
3,Blue Lake,231
4,Trinidad,184
5,Fortuna,178


... (26 outputs left out)


----------------------------------------------------------------------
PERMANENT ADDRESS AFTER CLEANING: permanent_region
----------------------------------------------------------------------
Repeated values and occurrence counts:


,permanent_region,occurrences
0,LA,1602
1,BayArea,1464
2,OutOfState,1425
3,Local,1233
4,NorCal,1124
5,SanDiego,785


All rows containing those repeated values:


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
17,POP_S_6830775,student,6600 Alliance Rd,Arcata,CA,95521,9799 Meadow Pl,Berkeley,CA,94704,BayArea
19,POP_S_6877850,student,7456 Baldwin St,Arcata,CA,95521,5309 Birch Dr,San Jose,CA,95112,BayArea
20,POP_S_6413414,student,5161 LK Wood Blvd,Arcata,CA,95521,8852 Maple Ln,Oakland,CA,94612,BayArea
22,POP_S_8903984,student,7959 Murray Rd,McKinleyville,CA,95519,8231 Lake St,Berkeley,CA,94704,BayArea
26,POP_S_5138664,student,2260 LK Wood Blvd,Arcata,CA,95521,2385 Ridge Pl,Oakland,CA,94612,BayArea
...,...,...,...,...,...,...,...,...,...,...,...
7602,POP_S_56063085,student,2905 St Louis Rd #501,Arcata,CA,95521,3605 Willow Way,Escondido,CA,92025,SanDiego
7623,POP_S_86622944,student,2905 St Louis Rd #311,Arcata,CA,95521,1496 Maple Way,Escondido,CA,92025,SanDiego
7628,POP_S_70346125,student,2905 St Louis Rd #708,Arcata,CA,95521,5102 Elm Blvd,National City,CA,91950,SanDiego
7629,POP_S_83353179,student,2905 St Louis Rd #258,Arcata,CA,95521,6387 Canyon Blvd,San Diego,CA,92103,SanDiego


In [17]:
# Final dataset audit

print("=" * 85)
print("FINAL DATA QUALITY AUDIT")
print("=" * 85)

# define expected structure

expected_columns = [
    "pop_id",
    "role",
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

missing_columns = [
    column for column in expected_columns
    if column not in pop.columns
]

unexpected_columns = [
    column for column in pop.columns
    if column not in expected_columns
]

print("\n1. DATASET STRUCTURE")
print("-" * 85)

print(f"Total records: {len(pop):,}")
print(f"Total columns: {len(pop.columns):,}")

if missing_columns:
    print(f"FAIL: Missing columns: {missing_columns}")
else:
    print("PASS: All expected columns are present.")

if unexpected_columns:
    print(f"WARNING: Unexpected columns: {unexpected_columns}")
else:
    print("PASS: No unexpected columns.")

# missing value audit

print("\n2. MISSING VALUE AUDIT")
print("-" * 85)

missing_summary = pd.DataFrame({
    "missing_values": pop.isna().sum(),
    "missing_percentage": (
        pop.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_values",
    ascending=False
)

display(missing_summary)

total_missing = int(pop.isna().sum().sum())

print(f"Total missing cells: {total_missing:,}")

# pop_id unique audit

print("\n3. POP_ID UNIQUENESS AUDIT")
print("-" * 85)

missing_ids = pop["pop_id"].isna().sum()

duplicate_ids = pop.loc[
    pop["pop_id"].notna()
    & pop["pop_id"].duplicated(keep=False)
].sort_values("pop_id")

print(f"Missing pop_id values: {missing_ids:,}")
print(f"Duplicate pop_id rows: {len(duplicate_ids):,}")
print(
    f"Unique duplicated identifiers: "
    f"{duplicate_ids['pop_id'].nunique():,}"
)

if duplicate_ids.empty:
    print("PASS: All pop_id values are unique.")
else:
    print("FAIL: Duplicate identifiers remain.")
    display(duplicate_ids)

# role and id_prefix audit

print("\n4. ROLE AND ID PREFIX AUDIT")
print("-" * 85)

role_prefix = {
    "Student": "POP_S",
    "Faculty": "POP_F",
    "Staff/Admin": "POP_SA",
    "Staff": "POP_ST"
}

# Normalize roles for validation without changing the DataFrame.
normalized_roles = (
    pop["role"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

role_mapping = {
    "student": "Student",
    "studnet": "Student",
    "faculty": "Faculty",
    "staff/admin": "Staff/Admin",
    "staff-admin": "Staff/Admin",
    "staff": "Staff"
}

standardized_roles = normalized_roles.map(role_mapping)

unexpected_roles = pop.loc[
    standardized_roles.isna(),
    ["pop_id", "role"]
]

if unexpected_roles.empty:
    print("PASS: All records have recognized roles.")
else:
    print("FAIL: Unexpected or missing roles found.")
    display(unexpected_roles)

# Check whether role labels are already standardized.
role_standardization_issues = pop.loc[
    standardized_roles.notna()
    & (
        pop["role"].astype("string").str.strip()
        != standardized_roles
    ),
    ["pop_id", "role"]
]

if role_standardization_issues.empty:
    print("PASS: Role labels are standardized.")
else:
    print("WARNING: Role labels require standardization.")
    display(role_standardization_issues)

# Validate the ID prefix associated with each role.
prefix_issues = []

for index, row in pop.iterrows():

    identifier = row["pop_id"]
    role = standardized_roles.loc[index]

    if pd.isna(identifier) or pd.isna(role):
        continue

    expected_prefix = role_prefix[role]

    expected_pattern = rf"^{re.escape(expected_prefix)}_\d+$"

    if not re.fullmatch(expected_pattern, str(identifier)):
        prefix_issues.append({
            "pop_id": identifier,
            "role": row["role"],
            "expected_prefix": expected_prefix
        })

prefix_issues = pd.DataFrame(prefix_issues)

if prefix_issues.empty:
    print("PASS: All IDs match their respective role prefixes.")
else:
    print("FAIL: Incorrect ID prefixes found.")
    display(prefix_issues)

# current city and zip audit

print("\n5. CURRENT ADDRESS AUDIT")
print("-" * 85)

city_zip_mapping = {
    "Arcata": "95521",
    "Eureka": "95501",
    "McKinleyville": "95519",
    "Blue Lake": "95525",
    "Trinidad": "95570",
    "Fortuna": "95540"
}

# Normalize values for validation.
current_city = (
    pop["current_city"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

current_zip = (
    pop["current_zip"]
    .astype("string")
    .str.strip()
)

city_lookup = {
    city.casefold(): zip_code
    for city, zip_code in city_zip_mapping.items()
}

# Identify city/ZIP inconsistencies.
current_address_issues = []

for index in pop.index:

    city = current_city.loc[index]
    zip_code = current_zip.loc[index]

    if pd.isna(city) or pd.isna(zip_code):
        continue

    if city in city_lookup:

        expected_zip = city_lookup[city]

        if zip_code != expected_zip:
            current_address_issues.append({
                "pop_id": pop.loc[index, "pop_id"],
                "current_city": pop.loc[index, "current_city"],
                "current_zip": pop.loc[index, "current_zip"],
                "expected_zip": expected_zip
            })

current_address_issues = pd.DataFrame(
    current_address_issues
)

if current_address_issues.empty:
    print("PASS: No city/ZIP inconsistencies found.")
else:
    print("FAIL: Current city and ZIP combinations conflict.")
    display(current_address_issues)

# current and permanent state audit

print("\n6. STATE AUDIT")
print("-" * 85)

valid_states = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE",
    "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS",
    "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS",
    "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY",
    "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV",
    "WI", "WY"
}

state_issues = []

for column in ["current_state", "permanent_state"]:

    normalized_state = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    invalid_mask = (
        normalized_state.notna()
        & ~normalized_state.isin(valid_states)
    )

    for index in pop.index[invalid_mask]:

        state_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

state_issues = pd.DataFrame(state_issues)

if state_issues.empty:
    print("PASS: All populated state values are valid abbreviations.")
else:
    print("WARNING: Invalid state values found.")
    display(state_issues)

# Verify the six designated current cities have California state.
california_cities = [
    city.casefold()
    for city in city_zip_mapping
]

california_state_issues = pop.loc[
    current_city.isin(california_cities)
    & (
        pop["current_state"]
        .astype("string")
        .str.strip()
        .str.upper()
        .ne("CA")
    ),
    ["pop_id", "current_city", "current_state"]
]

if california_state_issues.empty:
    print("PASS: Designated current cities have state CA.")
else:
    print("FAIL: Some designated cities do not have state CA.")
    display(california_state_issues)

# zip code format audit

print("\n7. ZIP CODE FORMAT AUDIT")
print("-" * 85)

zip_issues = []

for column in ["current_zip", "permanent_zip"]:

    normalized_zip = (
        pop[column]
        .astype("string")
        .str.strip()
    )

    invalid_mask = (
        normalized_zip.notna()
        & ~normalized_zip.str.fullmatch(r"\d{5}", na=False)
    )

    for index in pop.index[invalid_mask]:

        zip_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

zip_issues = pd.DataFrame(zip_issues)

if zip_issues.empty:
    print("PASS: All populated ZIP codes contain five digits.")
else:
    print("WARNING: Invalid ZIP formats found.")
    display(zip_issues)

# exact duplicae audit

print("\n8. EXACT DUPLICATE RECORD AUDIT")
print("-" * 85)

exact_duplicate_mask = pop.duplicated(keep=False)

exact_duplicates = pop.loc[
    exact_duplicate_mask
].sort_values("pop_id")

print(f"Rows involved in exact duplicates: {len(exact_duplicates):,}")

if exact_duplicates.empty:
    print("PASS: No completely identical records found.")
else:
    print("WARNING: Completely identical records found.")
    display(exact_duplicates)

# address completed summary

print("\n9. ADDRESS COMPLETENESS AUDIT")
print("-" * 85)

address_columns = [
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

address_completeness = pd.DataFrame({
    "populated_records": pop[address_columns].notna().sum(),
    "missing_records": pop[address_columns].isna().sum(),
    "completeness_percentage": (
        pop[address_columns].notna().mean() * 100
    ).round(2)
})

display(address_completeness)

# final readiness determination

print("\n" + "=" * 85)
print("FINAL DATASET READINESS")
print("=" * 85)

critical_checks = {
    "Expected columns present": len(missing_columns) == 0,
    "No missing pop_id values": missing_ids == 0,
    "Unique pop_id values": duplicate_ids.empty,
    "Recognized roles": unexpected_roles.empty,
    "Correct ID prefixes": prefix_issues.empty,
    "Current city/ZIP consistency": current_address_issues.empty,
    "Valid state abbreviations": state_issues.empty,
    "California city/state consistency": california_state_issues.empty,
    "Valid ZIP formats": zip_issues.empty
}

for check, passed in critical_checks.items():

    status = "PASS" if passed else "FAIL"

    print(f"{status}: {check}")

all_critical_checks_passed = all(critical_checks.values())

print("\n" + "-" * 85)

if all_critical_checks_passed:

    print("RESULT: DATASET PASSES ALL CRITICAL VALIDATION CHECKS.")

    if total_missing > 0:
        print(
            "NOTE: Missing values remain. Review the completeness "
            "report and determine whether they are acceptable."
        )

    if not exact_duplicates.empty:
        print(
            "NOTE: Exact duplicate records remain. "
            "Review them before finalizing the dataset."
        )

else:

    print("RESULT: DATASET IS NOT READY FOR FINAL EXPORT.")
    print("Resolve the failed validation checks before proceeding.")

print("=" * 85)

FINAL DATA QUALITY AUDIT

1. DATASET STRUCTURE
-------------------------------------------------------------------------------------
Total records: 7,633
Total columns: 11
PASS: All expected columns are present.
PASS: No unexpected columns.

2. MISSING VALUE AUDIT
-------------------------------------------------------------------------------------


,missing_values,missing_percentage
permanent_street,111,1.45
current_street,101,1.32
pop_id,0,0.00
role,0,0.00
current_city,0,0.00
current_state,0,0.00
current_zip,0,0.00
permanent_city,0,0.00
permanent_state,0,0.00
permanent_zip,0,0.00


Total missing cells: 212

3. POP_ID UNIQUENESS AUDIT
-------------------------------------------------------------------------------------
Missing pop_id values: 0
Duplicate pop_id rows: 8
Unique duplicated identifiers: 4
FAIL: Duplicate identifiers remain.


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
1427,POP_S_1371371,student,5576 G St,Arcata,CA,95521,282 Hill Ln,Glendale,CA,91203,LA
7163,POP_S_1371371,staff/admin,2570 Myrtle Ave,Eureka,CA,95501,2570 Myrtle Ave,Eureka,CA,95501,Local
380,POP_S_4940241,student,8640 Janis Ct,Arcata,CA,95521,7386 Lake Rd,Chico,CA,95926,NorCal
4552,POP_S_4940241,student,1466 Spears Rd,Arcata,CA,95521,6155 Spruce Dr,San Diego,CA,92103,SanDiego
3880,POP_S_6159896,student,1695 G St,Arcata,CA,95521,2144 Willow Way,Austin,TX,78701,OutOfState
7097,POP_S_6159896,staff/admin,8317 WINCHESTER AVE,McKinleyville,CA,95519,5317 Winchester Ave,McKinleyville,CA,95519,Local
43,POP_S_7366358,student,2208 Baldwin St,Arcata,CA,95521,9805 Meadow Blvd,Fremont,CA,94536,BayArea
3554,POP_S_7366358,student,2551 Union St,Arcata,CA,95521,421 Lake Rd,Santa Rosa,CA,95401,NorCal



4. ROLE AND ID PREFIX AUDIT
-------------------------------------------------------------------------------------
PASS: All records have recognized roles.


,pop_id,role
0,POP_S_7315039,student
1,POP_S_4194026,student
2,POP_S_4478952,student
3,POP_S_2541649,student
4,POP_S_4662231,student
...,...,...
7628,POP_S_70346125,student
7629,POP_S_83353179,student
7630,POP_S_42087782,student
7631,POP_S_93192061,student


FAIL: Incorrect ID prefixes found.


,pop_id,role,expected_prefix
0,POP_S_1717516,staff/admin,POP_SA
1,POP_S_1418572,staff/admin,POP_SA
2,POP_S_3328417,staff/admin,POP_SA
3,POP_S_6650936,staff/admin,POP_SA
4,POP_S_1657976,staff/admin,POP_SA
...,...,...,...
708,POP_S_4593087,staff/admin,POP_SA
709,POP_S_6591844,staff/admin,POP_SA
710,POP_S_2353753,staff/admin,POP_SA
711,POP_S_4832818,staff/admin,POP_SA



5. CURRENT ADDRESS AUDIT
-------------------------------------------------------------------------------------
PASS: No city/ZIP inconsistencies found.

6. STATE AUDIT
-------------------------------------------------------------------------------------
PASS: All populated state values are valid abbreviations.
PASS: Designated current cities have state CA.

7. ZIP CODE FORMAT AUDIT
-------------------------------------------------------------------------------------
PASS: All populated ZIP codes contain five digits.

8. EXACT DUPLICATE RECORD AUDIT
-------------------------------------------------------------------------------------
Rows involved in exact duplicates: 0
PASS: No completely identical records found.

9. ADDRESS COMPLETENESS AUDIT
-------------------------------------------------------------------------------------


,populated_records,missing_records,completeness_percentage
current_street,7532,101,98.68
current_city,7633,0,100.00
current_state,7633,0,100.00
current_zip,7633,0,100.00
permanent_street,7522,111,98.55
permanent_city,7633,0,100.00
permanent_state,7633,0,100.00
permanent_zip,7633,0,100.00
permanent_region,7633,0,100.00



FINAL DATASET READINESS
PASS: Expected columns present
PASS: No missing pop_id values
FAIL: Unique pop_id values
PASS: Recognized roles
FAIL: Correct ID prefixes
PASS: Current city/ZIP consistency
PASS: Valid state abbreviations
PASS: California city/state consistency
PASS: Valid ZIP formats

-------------------------------------------------------------------------------------
RESULT: DATASET IS NOT READY FOR FINAL EXPORT.
Resolve the failed validation checks before proceeding.


In [18]:
# define prefix per role

role_prefix = {
    "student": "POP_S",
    "faculty": "POP_F",
    "staff/admin": "POP_SA"
}

# Standardize pop_id as a string and remove surrounding spaces.
pop["pop_id"] = pop["pop_id"].astype("string").str.strip()

# ID incorrect prefix

positions_to_reassign = set()

for position in range(len(pop)):

    identifier = pop.iloc[position]["pop_id"]
    role = pop.iloc[position]["role"]
    expected_prefix = role_prefix[role]

    # Require the correct prefix and a numeric suffix.
    expected_pattern = rf"^{re.escape(expected_prefix)}_\d+$"

    if not re.fullmatch(expected_pattern, identifier):
        positions_to_reassign.add(position)

# generate unique replacement ID

# Track all existing identifiers to prevent collisions.
used_ids = set(pop["pop_id"].tolist())

# Find the largest existing numeric suffix for each prefix.
max_suffix = defaultdict(int)

for identifier in used_ids:

    match = re.fullmatch(r"^(POP_[A-Z]+)_(\d+)$", identifier)

    if match:
        prefix = match.group(1)
        suffix = int(match.group(2))
        max_suffix[prefix] = max(max_suffix[prefix], suffix)

# Allocate new IDs only to the records that need correction.
pop_id_column_position = pop.columns.get_loc("pop_id")

for position in sorted(positions_to_reassign):

    role = pop.iloc[position]["role"]
    prefix = role_prefix[role]

    # Start after the largest existing suffix for this prefix.
    candidate_number = max_suffix[prefix] + 1
    new_identifier = f"{prefix}_{candidate_number}"

    # Guarantee uniqueness.
    while new_identifier in used_ids:
        candidate_number += 1
        new_identifier = f"{prefix}_{candidate_number}"

    # Assign the replacement ID.
    pop.iat[position, pop_id_column_position] = new_identifier

    # Update tracking variables.
    used_ids.add(new_identifier)
    max_suffix[prefix] = candidate_number

In [19]:
# Final dataset audit

print("=" * 85)
print("FINAL DATA QUALITY AUDIT")
print("=" * 85)

# define expected structure

expected_columns = [
    "pop_id",
    "role",
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

missing_columns = [
    column for column in expected_columns
    if column not in pop.columns
]

unexpected_columns = [
    column for column in pop.columns
    if column not in expected_columns
]

print("\n1. DATASET STRUCTURE")
print("-" * 85)

print(f"Total records: {len(pop):,}")
print(f"Total columns: {len(pop.columns):,}")

if missing_columns:
    print(f"FAIL: Missing columns: {missing_columns}")
else:
    print("PASS: All expected columns are present.")

if unexpected_columns:
    print(f"WARNING: Unexpected columns: {unexpected_columns}")
else:
    print("PASS: No unexpected columns.")

# missing value audit

print("\n2. MISSING VALUE AUDIT")
print("-" * 85)

missing_summary = pd.DataFrame({
    "missing_values": pop.isna().sum(),
    "missing_percentage": (
        pop.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_values",
    ascending=False
)

display(missing_summary)

total_missing = int(pop.isna().sum().sum())

print(f"Total missing cells: {total_missing:,}")

# pop_id unique audit

print("\n3. POP_ID UNIQUENESS AUDIT")
print("-" * 85)

missing_ids = pop["pop_id"].isna().sum()

duplicate_ids = pop.loc[
    pop["pop_id"].notna()
    & pop["pop_id"].duplicated(keep=False)
].sort_values("pop_id")

print(f"Missing pop_id values: {missing_ids:,}")
print(f"Duplicate pop_id rows: {len(duplicate_ids):,}")
print(
    f"Unique duplicated identifiers: "
    f"{duplicate_ids['pop_id'].nunique():,}"
)

if duplicate_ids.empty:
    print("PASS: All pop_id values are unique.")
else:
    print("FAIL: Duplicate identifiers remain.")
    display(duplicate_ids)

# role and id_prefix audit

print("\n4. ROLE AND ID PREFIX AUDIT")
print("-" * 85)

role_prefix = {
    "Student": "POP_S",
    "Faculty": "POP_F",
    "Staff/Admin": "POP_SA",
    "Staff": "POP_ST"
}

# Normalize roles for validation without changing the DataFrame.
normalized_roles = (
    pop["role"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

role_mapping = {
    "student": "Student",
    "studnet": "Student",
    "faculty": "Faculty",
    "staff/admin": "Staff/Admin",
    "staff-admin": "Staff/Admin",
    "staff": "Staff"
}

standardized_roles = normalized_roles.map(role_mapping)

unexpected_roles = pop.loc[
    standardized_roles.isna(),
    ["pop_id", "role"]
]

if unexpected_roles.empty:
    print("PASS: All records have recognized roles.")
else:
    print("FAIL: Unexpected or missing roles found.")
    display(unexpected_roles)

# Check whether role labels are already standardized.
role_standardization_issues = pop.loc[
    standardized_roles.notna()
    & (
        pop["role"].astype("string").str.strip()
        != standardized_roles
    ),
    ["pop_id", "role"]
]

if role_standardization_issues.empty:
    print("PASS: Role labels are standardized.")
else:
    print("WARNING: Role labels require standardization.")
    display(role_standardization_issues)

# Validate the ID prefix associated with each role.
prefix_issues = []

for index, row in pop.iterrows():

    identifier = row["pop_id"]
    role = standardized_roles.loc[index]

    if pd.isna(identifier) or pd.isna(role):
        continue

    expected_prefix = role_prefix[role]

    expected_pattern = rf"^{re.escape(expected_prefix)}_\d+$"

    if not re.fullmatch(expected_pattern, str(identifier)):
        prefix_issues.append({
            "pop_id": identifier,
            "role": row["role"],
            "expected_prefix": expected_prefix
        })

prefix_issues = pd.DataFrame(prefix_issues)

if prefix_issues.empty:
    print("PASS: All IDs match their respective role prefixes.")
else:
    print("FAIL: Incorrect ID prefixes found.")
    display(prefix_issues)

# current city and zip audit

print("\n5. CURRENT ADDRESS AUDIT")
print("-" * 85)

city_zip_mapping = {
    "Arcata": "95521",
    "Eureka": "95501",
    "McKinleyville": "95519",
    "Blue Lake": "95525",
    "Trinidad": "95570",
    "Fortuna": "95540"
}

# Normalize values for validation.
current_city = (
    pop["current_city"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

current_zip = (
    pop["current_zip"]
    .astype("string")
    .str.strip()
)

city_lookup = {
    city.casefold(): zip_code
    for city, zip_code in city_zip_mapping.items()
}

# Identify city/ZIP inconsistencies.
current_address_issues = []

for index in pop.index:

    city = current_city.loc[index]
    zip_code = current_zip.loc[index]

    if pd.isna(city) or pd.isna(zip_code):
        continue

    if city in city_lookup:

        expected_zip = city_lookup[city]

        if zip_code != expected_zip:
            current_address_issues.append({
                "pop_id": pop.loc[index, "pop_id"],
                "current_city": pop.loc[index, "current_city"],
                "current_zip": pop.loc[index, "current_zip"],
                "expected_zip": expected_zip
            })

current_address_issues = pd.DataFrame(
    current_address_issues
)

if current_address_issues.empty:
    print("PASS: No city/ZIP inconsistencies found.")
else:
    print("FAIL: Current city and ZIP combinations conflict.")
    display(current_address_issues)

# current and permanent state audit

print("\n6. STATE AUDIT")
print("-" * 85)

valid_states = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE",
    "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS",
    "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS",
    "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY",
    "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV",
    "WI", "WY"
}

state_issues = []

for column in ["current_state", "permanent_state"]:

    normalized_state = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    invalid_mask = (
        normalized_state.notna()
        & ~normalized_state.isin(valid_states)
    )

    for index in pop.index[invalid_mask]:

        state_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

state_issues = pd.DataFrame(state_issues)

if state_issues.empty:
    print("PASS: All populated state values are valid abbreviations.")
else:
    print("WARNING: Invalid state values found.")
    display(state_issues)

# Verify the six designated current cities have California state.
california_cities = [
    city.casefold()
    for city in city_zip_mapping
]

california_state_issues = pop.loc[
    current_city.isin(california_cities)
    & (
        pop["current_state"]
        .astype("string")
        .str.strip()
        .str.upper()
        .ne("CA")
    ),
    ["pop_id", "current_city", "current_state"]
]

if california_state_issues.empty:
    print("PASS: Designated current cities have state CA.")
else:
    print("FAIL: Some designated cities do not have state CA.")
    display(california_state_issues)

# zip code format audit

print("\n7. ZIP CODE FORMAT AUDIT")
print("-" * 85)

zip_issues = []

for column in ["current_zip", "permanent_zip"]:

    normalized_zip = (
        pop[column]
        .astype("string")
        .str.strip()
    )

    invalid_mask = (
        normalized_zip.notna()
        & ~normalized_zip.str.fullmatch(r"\d{5}", na=False)
    )

    for index in pop.index[invalid_mask]:

        zip_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

zip_issues = pd.DataFrame(zip_issues)

if zip_issues.empty:
    print("PASS: All populated ZIP codes contain five digits.")
else:
    print("WARNING: Invalid ZIP formats found.")
    display(zip_issues)

# exact duplicae audit

print("\n8. EXACT DUPLICATE RECORD AUDIT")
print("-" * 85)

exact_duplicate_mask = pop.duplicated(keep=False)

exact_duplicates = pop.loc[
    exact_duplicate_mask
].sort_values("pop_id")

print(f"Rows involved in exact duplicates: {len(exact_duplicates):,}")

if exact_duplicates.empty:
    print("PASS: No completely identical records found.")
else:
    print("WARNING: Completely identical records found.")
    display(exact_duplicates)

# address completed summary

print("\n9. ADDRESS COMPLETENESS AUDIT")
print("-" * 85)

address_columns = [
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

address_completeness = pd.DataFrame({
    "populated_records": pop[address_columns].notna().sum(),
    "missing_records": pop[address_columns].isna().sum(),
    "completeness_percentage": (
        pop[address_columns].notna().mean() * 100
    ).round(2)
})

display(address_completeness)

# final readiness determination

print("\n" + "=" * 85)
print("FINAL DATASET READINESS")
print("=" * 85)

critical_checks = {
    "Expected columns present": len(missing_columns) == 0,
    "No missing pop_id values": missing_ids == 0,
    "Unique pop_id values": duplicate_ids.empty,
    "Recognized roles": unexpected_roles.empty,
    "Correct ID prefixes": prefix_issues.empty,
    "Current city/ZIP consistency": current_address_issues.empty,
    "Valid state abbreviations": state_issues.empty,
    "California city/state consistency": california_state_issues.empty,
    "Valid ZIP formats": zip_issues.empty
}

for check, passed in critical_checks.items():

    status = "PASS" if passed else "FAIL"

    print(f"{status}: {check}")

all_critical_checks_passed = all(critical_checks.values())

print("\n" + "-" * 85)

if all_critical_checks_passed:

    print("RESULT: DATASET PASSES ALL CRITICAL VALIDATION CHECKS.")

    if total_missing > 0:
        print(
            "NOTE: Missing values remain. Review the completeness "
            "report and determine whether they are acceptable."
        )

    if not exact_duplicates.empty:
        print(
            "NOTE: Exact duplicate records remain. "
            "Review them before finalizing the dataset."
        )

else:

    print("RESULT: DATASET IS NOT READY FOR FINAL EXPORT.")
    print("Resolve the failed validation checks before proceeding.")

print("=" * 85)

FINAL DATA QUALITY AUDIT

1. DATASET STRUCTURE
-------------------------------------------------------------------------------------
Total records: 7,633
Total columns: 11
PASS: All expected columns are present.
PASS: No unexpected columns.

2. MISSING VALUE AUDIT
-------------------------------------------------------------------------------------


,missing_values,missing_percentage
permanent_street,111,1.45
current_street,101,1.32
pop_id,0,0.00
role,0,0.00
current_city,0,0.00
current_state,0,0.00
current_zip,0,0.00
permanent_city,0,0.00
permanent_state,0,0.00
permanent_zip,0,0.00


Total missing cells: 212

3. POP_ID UNIQUENESS AUDIT
-------------------------------------------------------------------------------------
Missing pop_id values: 0
Duplicate pop_id rows: 4
Unique duplicated identifiers: 2
FAIL: Duplicate identifiers remain.


,pop_id,role,current_street,current_city,current_state,current_zip,permanent_street,permanent_city,permanent_state,permanent_zip,permanent_region
380,POP_S_4940241,student,8640 Janis Ct,Arcata,CA,95521,7386 Lake Rd,Chico,CA,95926,NorCal
4552,POP_S_4940241,student,1466 Spears Rd,Arcata,CA,95521,6155 Spruce Dr,San Diego,CA,92103,SanDiego
43,POP_S_7366358,student,2208 Baldwin St,Arcata,CA,95521,9805 Meadow Blvd,Fremont,CA,94536,BayArea
3554,POP_S_7366358,student,2551 Union St,Arcata,CA,95521,421 Lake Rd,Santa Rosa,CA,95401,NorCal



4. ROLE AND ID PREFIX AUDIT
-------------------------------------------------------------------------------------
PASS: All records have recognized roles.


,pop_id,role
0,POP_S_7315039,student
1,POP_S_4194026,student
2,POP_S_4478952,student
3,POP_S_2541649,student
4,POP_S_4662231,student
...,...,...
7628,POP_S_70346125,student
7629,POP_S_83353179,student
7630,POP_S_42087782,student
7631,POP_S_93192061,student


PASS: All IDs match their respective role prefixes.

5. CURRENT ADDRESS AUDIT
-------------------------------------------------------------------------------------
PASS: No city/ZIP inconsistencies found.

6. STATE AUDIT
-------------------------------------------------------------------------------------
PASS: All populated state values are valid abbreviations.
PASS: Designated current cities have state CA.

7. ZIP CODE FORMAT AUDIT
-------------------------------------------------------------------------------------
PASS: All populated ZIP codes contain five digits.

8. EXACT DUPLICATE RECORD AUDIT
-------------------------------------------------------------------------------------
Rows involved in exact duplicates: 0
PASS: No completely identical records found.

9. ADDRESS COMPLETENESS AUDIT
-------------------------------------------------------------------------------------


,populated_records,missing_records,completeness_percentage
current_street,7532,101,98.68
current_city,7633,0,100.00
current_state,7633,0,100.00
current_zip,7633,0,100.00
permanent_street,7522,111,98.55
permanent_city,7633,0,100.00
permanent_state,7633,0,100.00
permanent_zip,7633,0,100.00
permanent_region,7633,0,100.00



FINAL DATASET READINESS
PASS: Expected columns present
PASS: No missing pop_id values
FAIL: Unique pop_id values
PASS: Recognized roles
PASS: Correct ID prefixes
PASS: Current city/ZIP consistency
PASS: Valid state abbreviations
PASS: California city/state consistency
PASS: Valid ZIP formats

-------------------------------------------------------------------------------------
RESULT: DATASET IS NOT READY FOR FINAL EXPORT.
Resolve the failed validation checks before proceeding.


In [20]:
# assign new pop_id to duplicates.

# Preserve a backup before modifying identifiers.
pop_before_reassignment = pop.copy(deep=True)

# Ensure pop_id is stored as a string.
pop["pop_id"] = pop["pop_id"].astype("string").str.strip()

# Identify existing IDs and their numeric suffixes.
used_ids = set(pop["pop_id"].dropna())

student_ids = [
    int(match.group(1))
    for identifier in used_ids
    if (match := re.fullmatch(r"POP_S_(\d+)", identifier))
]

# Start after the largest existing student ID.
next_id = max(student_ids, default=0) + 1

# Identify duplicate IDs.
duplicate_mask = pop["pop_id"].duplicated(keep="first")

# Store the original identifiers for comparison.
original_ids = pop["pop_id"].copy()

# Assign new IDs to subsequent duplicate records.
for index in pop.index[duplicate_mask]:

    new_id = f"POP_S_{next_id}"

    # Ensure the new ID does not already exist.
    while new_id in used_ids:
        next_id += 1
        new_id = f"POP_S_{next_id}"

    # Assign the new identifier.
    pop.at[index, "pop_id"] = new_id

    # Track the new identifier.
    used_ids.add(new_id)
    next_id += 1

# show outputs

changed_records = pop.loc[
    original_ids != pop["pop_id"],
    ["pop_id", "role", "current_street",
     "current_city", "permanent_street", "permanent_city"]
].copy()

changed_records.insert(
    0,
    "original_pop_id",
    original_ids.loc[original_ids != pop["pop_id"]]
)

print("=" * 70)
print("POP_ID REASSIGNMENT RESULTS")
print("=" * 70)

print(f"New identifiers assigned: {len(changed_records)}")

display(changed_records)

# verify if unique

print("\nFINAL POP_ID VALIDATION")
print("-" * 70)

print(f"Total records: {len(pop):,}")
print(f"Missing IDs: {pop['pop_id'].isna().sum():,}")
print(f"Duplicate IDs: {pop['pop_id'].duplicated().sum():,}")

assert pop["pop_id"].notna().all(), "Missing IDs remain."
assert pop["pop_id"].is_unique, "Duplicate IDs remain."

print("\nPASS: All pop_id values are unique.")

POP_ID REASSIGNMENT RESULTS
New identifiers assigned: 2


,original_pop_id,pop_id,role,current_street,current_city,permanent_street,permanent_city
3554,POP_S_7366358,POP_S_99263434,student,2551 Union St,Arcata,421 Lake Rd,Santa Rosa
4552,POP_S_4940241,POP_S_99263435,student,1466 Spears Rd,Arcata,6155 Spruce Dr,San Diego



FINAL POP_ID VALIDATION
----------------------------------------------------------------------
Total records: 7,633
Missing IDs: 0
Duplicate IDs: 0

PASS: All pop_id values are unique.


In [21]:
# Final dataset audit

print("=" * 85)
print("FINAL DATA QUALITY AUDIT")
print("=" * 85)

# define expected structure

expected_columns = [
    "pop_id",
    "role",
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

missing_columns = [
    column for column in expected_columns
    if column not in pop.columns
]

unexpected_columns = [
    column for column in pop.columns
    if column not in expected_columns
]

print("\n1. DATASET STRUCTURE")
print("-" * 85)

print(f"Total records: {len(pop):,}")
print(f"Total columns: {len(pop.columns):,}")

if missing_columns:
    print(f"FAIL: Missing columns: {missing_columns}")
else:
    print("PASS: All expected columns are present.")

if unexpected_columns:
    print(f"WARNING: Unexpected columns: {unexpected_columns}")
else:
    print("PASS: No unexpected columns.")

# missing value audit

print("\n2. MISSING VALUE AUDIT")
print("-" * 85)

missing_summary = pd.DataFrame({
    "missing_values": pop.isna().sum(),
    "missing_percentage": (
        pop.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "missing_values",
    ascending=False
)

display(missing_summary)

total_missing = int(pop.isna().sum().sum())

print(f"Total missing cells: {total_missing:,}")

# pop_id unique audit

print("\n3. POP_ID UNIQUENESS AUDIT")
print("-" * 85)

missing_ids = pop["pop_id"].isna().sum()

duplicate_ids = pop.loc[
    pop["pop_id"].notna()
    & pop["pop_id"].duplicated(keep=False)
].sort_values("pop_id")

print(f"Missing pop_id values: {missing_ids:,}")
print(f"Duplicate pop_id rows: {len(duplicate_ids):,}")
print(
    f"Unique duplicated identifiers: "
    f"{duplicate_ids['pop_id'].nunique():,}"
)

if duplicate_ids.empty:
    print("PASS: All pop_id values are unique.")
else:
    print("FAIL: Duplicate identifiers remain.")
    display(duplicate_ids)

# role and id_prefix audit

print("\n4. ROLE AND ID PREFIX AUDIT")
print("-" * 85)

role_prefix = {
    "Student": "POP_S",
    "Faculty": "POP_F",
    "Staff/Admin": "POP_SA",
    "Staff": "POP_ST"
}

# Normalize roles for validation without changing the DataFrame.
normalized_roles = (
    pop["role"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

role_mapping = {
    "student": "Student",
    "studnet": "Student",
    "faculty": "Faculty",
    "staff/admin": "Staff/Admin",
    "staff-admin": "Staff/Admin",
    "staff": "Staff"
}

standardized_roles = normalized_roles.map(role_mapping)

unexpected_roles = pop.loc[
    standardized_roles.isna(),
    ["pop_id", "role"]
]

if unexpected_roles.empty:
    print("PASS: All records have recognized roles.")
else:
    print("FAIL: Unexpected or missing roles found.")
    display(unexpected_roles)

# Check whether role labels are already standardized.
role_standardization_issues = pop.loc[
    standardized_roles.notna()
    & (
        pop["role"].astype("string").str.strip()
        != standardized_roles
    ),
    ["pop_id", "role"]
]

if role_standardization_issues.empty:
    print("PASS: Role labels are standardized.")
else:
    print("WARNING: Role labels require standardization.")
    display(role_standardization_issues)

# Validate the ID prefix associated with each role.
prefix_issues = []

for index, row in pop.iterrows():

    identifier = row["pop_id"]
    role = standardized_roles.loc[index]

    if pd.isna(identifier) or pd.isna(role):
        continue

    expected_prefix = role_prefix[role]

    expected_pattern = rf"^{re.escape(expected_prefix)}_\d+$"

    if not re.fullmatch(expected_pattern, str(identifier)):
        prefix_issues.append({
            "pop_id": identifier,
            "role": row["role"],
            "expected_prefix": expected_prefix
        })

prefix_issues = pd.DataFrame(prefix_issues)

if prefix_issues.empty:
    print("PASS: All IDs match their respective role prefixes.")
else:
    print("FAIL: Incorrect ID prefixes found.")
    display(prefix_issues)

# current city and zip audit

print("\n5. CURRENT ADDRESS AUDIT")
print("-" * 85)

city_zip_mapping = {
    "Arcata": "95521",
    "Eureka": "95501",
    "McKinleyville": "95519",
    "Blue Lake": "95525",
    "Trinidad": "95570",
    "Fortuna": "95540"
}

# Normalize values for validation.
current_city = (
    pop["current_city"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

current_zip = (
    pop["current_zip"]
    .astype("string")
    .str.strip()
)

city_lookup = {
    city.casefold(): zip_code
    for city, zip_code in city_zip_mapping.items()
}

# Identify city/ZIP inconsistencies.
current_address_issues = []

for index in pop.index:

    city = current_city.loc[index]
    zip_code = current_zip.loc[index]

    if pd.isna(city) or pd.isna(zip_code):
        continue

    if city in city_lookup:

        expected_zip = city_lookup[city]

        if zip_code != expected_zip:
            current_address_issues.append({
                "pop_id": pop.loc[index, "pop_id"],
                "current_city": pop.loc[index, "current_city"],
                "current_zip": pop.loc[index, "current_zip"],
                "expected_zip": expected_zip
            })

current_address_issues = pd.DataFrame(
    current_address_issues
)

if current_address_issues.empty:
    print("PASS: No city/ZIP inconsistencies found.")
else:
    print("FAIL: Current city and ZIP combinations conflict.")
    display(current_address_issues)

# current and permanent state audit

print("\n6. STATE AUDIT")
print("-" * 85)

valid_states = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE",
    "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS",
    "KY", "LA", "ME", "MD", "MA", "MI", "MN", "MS",
    "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY",
    "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV",
    "WI", "WY"
}

state_issues = []

for column in ["current_state", "permanent_state"]:

    normalized_state = (
        pop[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    invalid_mask = (
        normalized_state.notna()
        & ~normalized_state.isin(valid_states)
    )

    for index in pop.index[invalid_mask]:

        state_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

state_issues = pd.DataFrame(state_issues)

if state_issues.empty:
    print("PASS: All populated state values are valid abbreviations.")
else:
    print("WARNING: Invalid state values found.")
    display(state_issues)

# Verify the six designated current cities have California state.
california_cities = [
    city.casefold()
    for city in city_zip_mapping
]

california_state_issues = pop.loc[
    current_city.isin(california_cities)
    & (
        pop["current_state"]
        .astype("string")
        .str.strip()
        .str.upper()
        .ne("CA")
    ),
    ["pop_id", "current_city", "current_state"]
]

if california_state_issues.empty:
    print("PASS: Designated current cities have state CA.")
else:
    print("FAIL: Some designated cities do not have state CA.")
    display(california_state_issues)

# zip code format audit

print("\n7. ZIP CODE FORMAT AUDIT")
print("-" * 85)

zip_issues = []

for column in ["current_zip", "permanent_zip"]:

    normalized_zip = (
        pop[column]
        .astype("string")
        .str.strip()
    )

    invalid_mask = (
        normalized_zip.notna()
        & ~normalized_zip.str.fullmatch(r"\d{5}", na=False)
    )

    for index in pop.index[invalid_mask]:

        zip_issues.append({
            "pop_id": pop.loc[index, "pop_id"],
            "column": column,
            "value": pop.loc[index, column]
        })

zip_issues = pd.DataFrame(zip_issues)

if zip_issues.empty:
    print("PASS: All populated ZIP codes contain five digits.")
else:
    print("WARNING: Invalid ZIP formats found.")
    display(zip_issues)

# exact duplicae audit

print("\n8. EXACT DUPLICATE RECORD AUDIT")
print("-" * 85)

exact_duplicate_mask = pop.duplicated(keep=False)

exact_duplicates = pop.loc[
    exact_duplicate_mask
].sort_values("pop_id")

print(f"Rows involved in exact duplicates: {len(exact_duplicates):,}")

if exact_duplicates.empty:
    print("PASS: No completely identical records found.")
else:
    print("WARNING: Completely identical records found.")
    display(exact_duplicates)

# address completed summary

print("\n9. ADDRESS COMPLETENESS AUDIT")
print("-" * 85)

address_columns = [
    "current_street",
    "current_city",
    "current_state",
    "current_zip",
    "permanent_street",
    "permanent_city",
    "permanent_state",
    "permanent_zip",
    "permanent_region"
]

address_completeness = pd.DataFrame({
    "populated_records": pop[address_columns].notna().sum(),
    "missing_records": pop[address_columns].isna().sum(),
    "completeness_percentage": (
        pop[address_columns].notna().mean() * 100
    ).round(2)
})

display(address_completeness)

# final readiness determination

print("\n" + "=" * 85)
print("FINAL DATASET READINESS")
print("=" * 85)

critical_checks = {
    "Expected columns present": len(missing_columns) == 0,
    "No missing pop_id values": missing_ids == 0,
    "Unique pop_id values": duplicate_ids.empty,
    "Recognized roles": unexpected_roles.empty,
    "Correct ID prefixes": prefix_issues.empty,
    "Current city/ZIP consistency": current_address_issues.empty,
    "Valid state abbreviations": state_issues.empty,
    "California city/state consistency": california_state_issues.empty,
    "Valid ZIP formats": zip_issues.empty
}

for check, passed in critical_checks.items():

    status = "PASS" if passed else "FAIL"

    print(f"{status}: {check}")

all_critical_checks_passed = all(critical_checks.values())

print("\n" + "-" * 85)

if all_critical_checks_passed:

    print("RESULT: DATASET PASSES ALL CRITICAL VALIDATION CHECKS.")

    if total_missing > 0:
        print(
            "NOTE: Missing values remain. Review the completeness "
            "report and determine whether they are acceptable."
        )

    if not exact_duplicates.empty:
        print(
            "NOTE: Exact duplicate records remain. "
            "Review them before finalizing the dataset."
        )

else:

    print("RESULT: DATASET IS NOT READY FOR FINAL EXPORT.")
    print("Resolve the failed validation checks before proceeding.")

print("=" * 85)

FINAL DATA QUALITY AUDIT

1. DATASET STRUCTURE
-------------------------------------------------------------------------------------
Total records: 7,633
Total columns: 11
PASS: All expected columns are present.
PASS: No unexpected columns.

2. MISSING VALUE AUDIT
-------------------------------------------------------------------------------------


,missing_values,missing_percentage
permanent_street,111,1.45
current_street,101,1.32
pop_id,0,0.00
role,0,0.00
current_city,0,0.00
current_state,0,0.00
current_zip,0,0.00
permanent_city,0,0.00
permanent_state,0,0.00
permanent_zip,0,0.00


Total missing cells: 212

3. POP_ID UNIQUENESS AUDIT
-------------------------------------------------------------------------------------
Missing pop_id values: 0
Duplicate pop_id rows: 0
Unique duplicated identifiers: 0
PASS: All pop_id values are unique.

4. ROLE AND ID PREFIX AUDIT
-------------------------------------------------------------------------------------
PASS: All records have recognized roles.


,pop_id,role
0,POP_S_7315039,student
1,POP_S_4194026,student
2,POP_S_4478952,student
3,POP_S_2541649,student
4,POP_S_4662231,student
...,...,...
7628,POP_S_70346125,student
7629,POP_S_83353179,student
7630,POP_S_42087782,student
7631,POP_S_93192061,student


PASS: All IDs match their respective role prefixes.

5. CURRENT ADDRESS AUDIT
-------------------------------------------------------------------------------------
PASS: No city/ZIP inconsistencies found.

6. STATE AUDIT
-------------------------------------------------------------------------------------
PASS: All populated state values are valid abbreviations.
PASS: Designated current cities have state CA.

7. ZIP CODE FORMAT AUDIT
-------------------------------------------------------------------------------------
PASS: All populated ZIP codes contain five digits.

8. EXACT DUPLICATE RECORD AUDIT
-------------------------------------------------------------------------------------
Rows involved in exact duplicates: 0
PASS: No completely identical records found.

9. ADDRESS COMPLETENESS AUDIT
-------------------------------------------------------------------------------------


,populated_records,missing_records,completeness_percentage
current_street,7532,101,98.68
current_city,7633,0,100.00
current_state,7633,0,100.00
current_zip,7633,0,100.00
permanent_street,7522,111,98.55
permanent_city,7633,0,100.00
permanent_state,7633,0,100.00
permanent_zip,7633,0,100.00
permanent_region,7633,0,100.00



FINAL DATASET READINESS
PASS: Expected columns present
PASS: No missing pop_id values
PASS: Unique pop_id values
PASS: Recognized roles
PASS: Correct ID prefixes
PASS: Current city/ZIP consistency
PASS: Valid state abbreviations
PASS: California city/state consistency
PASS: Valid ZIP formats

-------------------------------------------------------------------------------------
RESULT: DATASET PASSES ALL CRITICAL VALIDATION CHECKS.
NOTE: Missing values remain. Review the completeness report and determine whether they are acceptable.


# Save the New Dataset

In [22]:
# Define the output directory
output_dir = Path(
    r"C:\Users\MARE\Documents\data311_project2\data\processed"
)

# Create the directory if it does not exist
output_dir.mkdir(parents=True, exist_ok=True)

# Define the output filename
output_file = output_dir / "population_addresses_cleaned_processed.csv"

# Save the cleaned DataFrame
pop.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully.")
print("File location:", output_file)
print("Number of records saved:", len(pop))

Cleaned dataset saved successfully.
File location: C:\Users\MARE\Documents\data311_project2\data\processed\population_addresses_cleaned_processed.csv
Number of records saved: 7633


# Commuter Data

In [23]:
com = pd.read_csv(r"C:\Users\MARE\OneDrive\Documents\data311_project2\data\raw\simulated_commuter_survey_2020_26_messy.csv")
com.head()

,respondent_id,year,role,consent,residence_type,days_commuting,trips_per_week,weeks_on_campus_cat,weeks_per_year,pct_walk_oncampus,pct_walk_offcampus,pct_tele,pct_drive,pct_shuttle,pct_carpool,pct_bus,pct_bike,pct_rideshare,pct_other,vehicle_type,ev_charge_offcampus,bike_power,secondary_mode,secondary_barrier,city,zip,street,nearest_intersection,resource_jackpass,resource_bikeshare,resource_blc,resource_ridematch,resource_zipcar,resource_express_bus,open_ended_no_car,distance_to_campus_miles,mode_primary,mtcde_est
0,S_02104,2020-21,Student,agree,"Off-campus residence that is not University housing (house, apartment, condo, trailer) within Hu...",4,8,37 weeks (two full semesters + one 5-week summer session),37.0,3.0,2,86.0,0,0.0,2.0,1.0,7.0,0.0,-1.0,Neither,NaN,"standard, non-electric",Vanpool,Flexibility,Eureka,95501,9485 K St,6th St & Myrtle Ave,knew_and_used,knew_did_not_use,"I knew about it, but I didn't use it.","I knew about it, but I didn't use it.","I didn't know about it, but now that I do I'll probably use it!","I didn't know about it, but now that I do I'll probably use it!",Having an e-bike would help.,6.85,tele,0.0315
1,S_00147,2020-21,Student,agree,"Off-campus residence that is not University housing (house, apartment, condo, trailer) within Hu...",6,NaN,40 weeks (two full semesters + one 8-week summer session),40.0,1.0,1,89.0,0,1.0,1.0,0.0,7.0,0.0,0.0,Plug-in electric vehicle,30-39%,"standard, non-electric",Personal electric bike or scooter,Flexibility,Arcata,95521,8849 H St,St Louis Rd #157 & St Louis Rd #435,knew_did_not_use,knew_did_not_use,knew_did_not_use,knew_did_not_use,did_not_know_wouldnt_use,did_not_know_might_use,NaN,0.98,tele,0.0052
2,S_00147,2020-21,Student,agree,"On-campus University housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)",6,12,32 weeks (two full semesters),32.0,2.0,1,87.0,0,6.0,2.0,0.0,2.0,0.0,0.0,Neither,NaN,electric,I have no other option,NaN,Arcata,95521,1474 14th St,LK Wood Blvd & St Louis Rd #431,knew_and_used,knew_did_not_use,knew_did_not_use,knew_did_not_use,knew_did_not_use,knew_and_used,NaN,0.42,walking,2.8e-05
3,S_01104,2020-21,Student,agree,"Off-campus residence that is not University housing (house, apartment, condo, trailer) within Hu...",5,10,32 weeks (two full semesters),32.0,0.0,6,87.0,1,0.0,1.0,0.0,4.0,1.0,0.0,Plug-in electric vehicle,70-79%,"standard, non-electric",Carpool/ride with others,Travel time,Arcata,95521,7942 Alliance Rd,St Louis Rd #108 & St Louis Rd #302,did_not_know_might_use,did_not_know_might_use,knew_did_not_use,knew_did_not_use,did_not_know_might_use,knew_did_not_use,Affordable housing closer to campus.,0.66,tele,0.002
4,S_01104,2020-21,Student,agree,"Off-campus residence that is not University housing (house, apartment, condo, trailer) within Hu...",5,10,32 weeks (two full semesters),32.0,2.0,2,87.0,2,0.0,1.0,0.0,4.0,2.0,0.0,Neither,NaN,"standard, non-electric",University shuttle,Safety,Arcata,95521,7302 H St,St Louis Rd #144 & St Louis Rd #426,knew_did_not_use,knew_and_used,did_not_know_wouldnt_use,did_not_know_wouldnt_use,did_not_know_might_use,did_not_know_might_use,NaN,1.06,tele,0.0058


In [24]:
# DATA AUDIT CONFIGURATION
 
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 100)

 
# INITIAL DATA SUMMARY
 

summary = pd.DataFrame({
    "unique_values": com.nunique(),
    "total_rows": len(com),
    "all_values_unique": com.apply(lambda col: col.is_unique),
    "missing_values": com.isna().sum()
})

print("=" * 80)
print("INITIAL DATA SUMMARY")
print("=" * 80)
display(summary)


 
# 1. DUPLICATE RESPONDENT ID AUDIT
 

print("\n" + "=" * 80)
print("1. DUPLICATE RESPONDENT_ID VALUES")
print("=" * 80)

duplicate_ids = com[
    com["respondent_id"].notna()
    & com["respondent_id"].duplicated(keep=False)
].sort_values("respondent_id")

if duplicate_ids.empty:
    print("No duplicate respondent_id values found.")
else:
    print("Repeated respondent IDs and their occurrence counts:")

    display(
        duplicate_ids["respondent_id"]
        .value_counts()
        .rename_axis("respondent_id")
        .reset_index(name="occurrences")
    )

    print("\nAll rows associated with repeated respondent IDs:")
    display(duplicate_ids)


 
# 2. ROLE AUDIT
 

print("\n" + "=" * 80)
print("2. ROLE AUDIT")
print("=" * 80)

print("All role values and their frequencies:")

role_counts = (
    com["role"]
    .value_counts(dropna=False)
    .rename_axis("role")
    .reset_index(name="count")
)

display(role_counts)

role_normalized = (
    com["role"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

print("\nRoles after ignoring capitalization and outer whitespace:")

display(
    com.assign(normalized_role=role_normalized)
    .groupby("normalized_role", dropna=False)
    .agg(
        count=("role", "size"),
        original_values=(
            "role",
            lambda x: sorted(x.dropna().unique().tolist())
        )
    )
    .reset_index()
)


 
# 3. YEAR AUDIT
 

print("\n" + "=" * 80)
print("3. YEAR AUDIT")
print("=" * 80)

print("All year values and their frequencies:")

year_counts = (
    com["year"]
    .value_counts(dropna=False)
    .rename_axis("year")
    .reset_index(name="count")
)

display(year_counts)

year_normalized = (
    com["year"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

print("\nYear values after ignoring whitespace and capitalization:")

display(
    com.assign(normalized_year=year_normalized)
    .groupby("normalized_year", dropna=False)
    .agg(
        count=("year", "size"),
        original_values=(
            "year",
            lambda x: sorted(x.dropna().unique().tolist())
        )
    )
    .reset_index()
)


 
# 4. RESIDENCE TYPE AUDIT
 

print("\n" + "=" * 80)
print("4. RESIDENCE TYPE AUDIT")
print("=" * 80)

print("All residence_type values and their frequencies:")

residence_counts = (
    com["residence_type"]
    .value_counts(dropna=False)
    .rename_axis("residence_type")
    .reset_index(name="count")
)

display(residence_counts)

residence_normalized = (
    com["residence_type"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

print("\nResidence types after ignoring capitalization and whitespace:")

display(
    com.assign(normalized_residence=residence_normalized)
    .groupby("normalized_residence", dropna=False)
    .agg(
        count=("residence_type", "size"),
        original_values=(
            "residence_type",
            lambda x: sorted(x.dropna().unique().tolist())
        )
    )
    .reset_index()
)


 
# 5. GEOGRAPHIC ADDRESS AUDIT
 

print("\n" + "=" * 80)
print("5. GEOGRAPHIC ADDRESS AUDIT")
print("=" * 80)

geographic_columns = [
    "city",
    "zip",
    "street",
    "nearest_intersection"
]


def audit_duplicates(df, columns, address_type):

    for column in columns:

        print("\n" + "-" * 70)
        print(f"{address_type}: {column}")
        print("-" * 70)

        counts = df[column].value_counts(dropna=True)
        repeated_values = counts[counts > 1]

        if repeated_values.empty:
            print("No repeated non-missing values found.")
            continue

        print("Repeated values and occurrence counts:")

        display(
            repeated_values
            .rename_axis(column)
            .reset_index(name="occurrences")
        )

        mask = (
            df[column].notna()
            & df[column].isin(repeated_values.index)
        )

        print("All rows containing those repeated values:")

        display(df.loc[mask].sort_values(column))


audit_duplicates(
    com,
    geographic_columns,
    "GEOGRAPHIC ADDRESS"
)


 
# 6. CASE AND WHITESPACE VARIATIONS
 

def audit_format_variations(df, columns, audit_type):

    for column in columns:

        normalized = (
            df[column]
            .astype("string")
            .str.strip()
            .str.casefold()
        )

        temp = pd.DataFrame({
            "normalized_value": normalized,
            "original_value": df[column]
        }).dropna(subset=["normalized_value"])

        variations = (
            temp.groupby("normalized_value")["original_value"]
            .agg(lambda x: sorted(x.unique().tolist()))
        )

        variations = variations[variations.map(len) > 1]

        if not variations.empty:

            print(
                f"\n{audit_type} FORMAT VARIATIONS: {column}"
            )

            display(
                variations
                .rename("original_variations")
                .reset_index()
            )


print("\n" + "=" * 80)
print("6. CASE AND WHITESPACE VARIATIONS")
print("=" * 80)

audit_format_variations(
    com,
    [
        "role",
        "year",
        "residence_type",
        "vehicle_type",
        "bike_power",
        "secondary_mode",
        "secondary_barrier",
        "city",
        "mode_primary"
    ] + geographic_columns,
    "TRANSPORTATION / GEOGRAPHIC"
)


 
# 7. MISSING VALUE AUDIT
 

print("\n" + "=" * 80)
print("7. MISSING VALUE AUDIT")
print("=" * 80)

missing_summary = pd.DataFrame({
    "missing_values": com.isna().sum(),
    "missing_percent": (
        com.isna().mean() * 100
    ).round(2),
    "unique_values": com.nunique()
})

missing_summary = missing_summary[
    missing_summary["missing_values"] > 0
].sort_values(
    "missing_values",
    ascending=False
)

if missing_summary.empty:
    print("No missing values found.")
else:
    print("Columns containing missing values:")
    display(missing_summary)


 
# 8. NUMERIC VARIABLE AUDIT
 

print("\n" + "=" * 80)
print("8. NUMERIC VARIABLE AUDIT")
print("=" * 80)

numeric_columns = [
    "days_commuting",
    "trips_per_week",
    "weeks_per_year",
    "pct_walk_oncampus",
    "pct_walk_offcampus",
    "pct_tele",
    "pct_drive",
    "pct_shuttle",
    "pct_carpool",
    "pct_bus",
    "pct_bike",
    "pct_rideshare",
    "pct_other",
    "distance_to_campus_miles",
    "mtcde_est"
]

numeric_columns = [
    col for col in numeric_columns
    if col in com.columns
]

numeric_audit = pd.DataFrame()

for column in numeric_columns:

    numeric_audit[column] = pd.to_numeric(
        com[column],
        errors="coerce"
    )

print("Numeric descriptive statistics:")
display(numeric_audit.describe().T)


# Identify values that cannot be converted to numeric
print("\nNon-numeric values in numeric columns:")

for column in numeric_columns:

    invalid = (
        com[column].notna()
        & pd.to_numeric(
            com[column],
            errors="coerce"
        ).isna()
    )

    if invalid.any():

        print(f"\nInvalid numeric values: {column}")

        display(
            com.loc[
                invalid,
                ["respondent_id", column]
            ]
        )


 
# 9. PERCENTAGE VALIDATION AUDIT
 

print("\n" + "=" * 80)
print("9. TRANSPORTATION PERCENTAGE VALIDATION")
print("=" * 80)

percentage_columns = [
    "pct_walk_oncampus",
    "pct_walk_offcampus",
    "pct_tele",
    "pct_drive",
    "pct_shuttle",
    "pct_carpool",
    "pct_bus",
    "pct_bike",
    "pct_rideshare",
    "pct_other"
]

percentage_columns = [
    col for col in percentage_columns
    if col in com.columns
]

percentage_data = com[percentage_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Check negative values and values greater than 100
for column in percentage_columns:

    invalid = (
        percentage_data[column].notna()
        & (
            (percentage_data[column] < 0)
            | (percentage_data[column] > 100)
        )
    )

    if invalid.any():

        print(f"\nInvalid percentage values: {column}")

        display(
            com.loc[
                invalid,
                ["respondent_id", column]
            ]
        )


# Check percentage totals only for complete, valid rows
complete_rows = percentage_data.notna().all(axis=1)

valid_rows = (
    complete_rows
    & percentage_data.ge(0).all(axis=1)
    & percentage_data.le(100).all(axis=1)
)

percentage_totals = percentage_data.sum(
    axis=1,
    min_count=len(percentage_columns)
)

invalid_totals = (
    valid_rows
    & ~percentage_totals.between(99, 101)
)

print("\nRecords with transportation percentages not totaling 100%:")

if invalid_totals.any():

    display(
        com.loc[
            invalid_totals,
            ["respondent_id"] + percentage_columns
        ].assign(
            percentage_total=percentage_totals[invalid_totals]
        )
    )

else:
    print("No percentage-total inconsistencies found among complete valid rows.")


 
# 10. DISTANCE AND GEOGRAPHIC VALIDATION
 

print("\n" + "=" * 80)
print("10. DISTANCE AND GEOGRAPHIC VALIDATION")
print("=" * 80)

distance_column = "distance_to_campus_miles"

if distance_column in com.columns:

    distance = pd.to_numeric(
        com[distance_column],
        errors="coerce"
    )

    print("Distance descriptive statistics:")
    display(distance.describe())

    invalid_distance = (
        distance.notna()
        & (distance < 0)
    )

    if invalid_distance.any():

        print("\nNegative distances detected:")

        display(
            com.loc[
                invalid_distance,
                ["respondent_id", distance_column]
            ]
        )

    print("\nDistance values equal to zero:")

    zero_distance = distance.eq(0)

    if zero_distance.any():

        display(
            com.loc[
                zero_distance,
                ["respondent_id", "city", "zip", distance_column]
            ]
        )
    else:
        print("No zero-distance values found.")


 
# 11. RESPONDENT-LEVEL CROSS-VARIABLE AUDIT
 

print("\n" + "=" * 80)
print("11. RESPONDENT-LEVEL CROSS-VARIABLE AUDIT")
print("=" * 80)

respondent_check_columns = [
    "year",
    "role",
    "consent",
    "residence_type",
    "city",
    "zip",
    "street",
    "mode_primary"
]

respondent_check_columns = [
    col for col in respondent_check_columns
    if col in com.columns
]

repeated_respondents = com[
    com["respondent_id"].notna()
    & com["respondent_id"].duplicated(keep=False)
]

if repeated_respondents.empty:

    print("No repeated respondents available for cross-variable review.")

else:

    print("Comparing repeated respondents across selected variables:")

    respondent_variation = (
        repeated_respondents
        .groupby("respondent_id")[respondent_check_columns]
        .nunique(dropna=True)
    )

    respondent_variation = respondent_variation[
        (respondent_variation > 1).any(axis=1)
    ]

    if respondent_variation.empty:

        print("No differences found in the selected variables.")

    else:

        print("Respondents with differing values:")
        display(respondent_variation)

        print("\nAssociated full records:")

        display(
            repeated_respondents[
                repeated_respondents["respondent_id"].isin(
                    respondent_variation.index
                )
            ].sort_values("respondent_id")
        )


 
# 12. DATA-TYPE AND IDENTIFIER AUDIT
 

print("\n" + "=" * 80)
print("12. DATA-TYPE AND IDENTIFIER AUDIT")
print("=" * 80)

print("Current column data types:")
display(com.dtypes.rename("data_type").to_frame())

print("\nRespondent ID formatting:")

id_values = com["respondent_id"].dropna().astype("string")

invalid_ids = ~id_values.str.fullmatch(
    r"S_\d+",
    na=False
)

if invalid_ids.any():

    print("Respondent IDs not following the expected S_#### pattern:")

    display(
        com.loc[
            com["respondent_id"].isin(
                id_values[invalid_ids]
            ),
            ["respondent_id", "year"]
        ]
    )

else:

    print("All non-missing respondent IDs follow the S_#### pattern.")


 
# 13. DATA-QUALITY AUDIT COMPLETE
 

print("\n" + "=" * 80)
print("DATA-QUALITY AUDIT COMPLETE")
print("=" * 80)


INITIAL DATA SUMMARY


,unique_values,total_rows,all_values_unique,missing_values
respondent_id,2693,7129,False,0
year,11,7129,False,0
role,9,7129,False,0
consent,1,7129,False,0
residence_type,9,7129,False,0
days_commuting,11,7129,False,0
trips_per_week,13,7129,False,168
weeks_on_campus_cat,20,7129,False,0
weeks_per_year,22,7129,False,0
pct_walk_oncampus,61,7129,False,0



1. DUPLICATE RESPONDENT_ID VALUES
Repeated respondent IDs and their occurrence counts:


,respondent_id,occurrences
0,S_00576,25
1,S_00905,21
2,F_02420,19
3,S_00147,19
4,S_01732,18
...,...,...
1588,S_02481,2
1589,S_02489,2
1590,S_02491,2
1591,S_02494,2



All rows associated with repeated respondent IDs:


,respondent_id,year,role,consent,residence_type,days_commuting,trips_per_week,weeks_on_campus_cat,weeks_per_year,pct_walk_oncampus,pct_walk_offcampus,pct_tele,pct_drive,pct_shuttle,pct_carpool,pct_bus,pct_bike,pct_rideshare,pct_other,vehicle_type,ev_charge_offcampus,bike_power,secondary_mode,secondary_barrier,city,zip,street,nearest_intersection,resource_jackpass,resource_bikeshare,resource_blc,resource_ridematch,resource_zipcar,resource_express_bus,open_ended_no_car,distance_to_campus_miles,mode_primary,mtcde_est
2994,F_00000,2022-23,Faculty,agree,"Off-campus residence that is not University housing (house, apartment, condo, trailer) within Hu...",6,12,44 weeks (two full semesters + one 8-week summer session),44.0,7.0,8,13.0,49,2.0,4.0,1.0,16.0,0.0,0.0,Motorcycle or moped,NaN,"standard, non-electric",Motorcycle/moped,Convenience,McKinleyville,95519,1461 Central Ave,Bates Rd & Winchester Ave,"I knew about it, but I didn't use it.","I knew about it, but I didn't use it.","I knew about it, but I didn't use it.",knew_did_not_use,"I knew about it, but I didn't use it.",knew_did_not_use,Weather makes biking hard in winter.,4.68,drive,0.5153
5382,F_00000,2024-25,Faculty,agree,"Off-campus residence that is not University housing (house, apartment, condo, trailer) within Hu...",5,10,"18 weeks (Faculty ""Green"" days for one full semester)",18.0,2.0,20,7.0,51,2.0,5.0,2.0,10.0,0.0,1.0,Neither,NaN,"standard, non-electric",Public transit,Convenience,McKinleyville,95519,5064 Hiller Rd,Central Ave & School Rd,knew_did_not_use,knew_did_not_use,did_not_know_might_use,knew_did_not_use,knew_did_not_use,did_not_know_wouldnt_use,More frequent buses aligned with class times.,4.87,drive,0.2091
6572,F_00004,2025-26,Faculty,agree,"On-campus University housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)",7,14,"18 weeks (Faculty ""Green"" days for one full semester)",18.0,5.0,14,12.0,0,1.0,0.0,0.0,12.0,0.0,0.0,Plug-in electric vehicle,91-100%,electric,University shuttle,NaN,Arcata,95521,1221 LK Wood Blvd,St Louis Rd #459 & St Louis Rd #611,knew_did_not_use,knew_did_not_use,did_not_know_wouldnt_use,knew_did_not_use,knew_did_not_use,did_not_know_might_use,NaN,0.48,drive,0.0606
6684,F_00004,2025-26,Faculty,agree,"Off-campus residence that is not University housing (house, apartment, condo, trailer) within Hu...",0,3,about a year,44.0,5.0,13,13.0,43,1.0,4.0,2.0,19.0,0.0,0.0,Plug-in electric vehicle,20-29%,electric,University shuttle,Flexibility,Arcata,95521,6442 G St,St Louis Rd #351 & St Louis Rd #519,did_not_know_wouldnt_use,knew_did_not_use,did_not_know_might_use,did_not_know_wouldnt_use,did_not_know_wouldnt_use,knew_did_not_use,More reliable evening transit.,0.61,DRIVE,0.0448
4153,F_00005,2023-24,Faculty,agree,"Off-campus residence that is not University housing (house, apartment, condo, trailer) within Hu...",5,10,"<18 weeks (less than Faculty ""Green"" days for one full semester)",14.0,4.0,17,12.0,47,0.0,5.0,5.0,10.0,0.0,0.0,Plug-in electric vehicle,40-49%,"standard, non-electric",Carpool/ride with others,Comfort,McKinleyville,95519,5516 Murray Rd,Bates Rd & School Rd,knew_and_used,knew_did_not_use,knew_did_not_use,did_not_know_might_use,did_not_know_might_use,knew_did_not_use,NaN,5.85,drive,0.174
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6028,S_02498,2025-26,Student,agree,"Off-campus residence that is not University housing (house, apartment, condo, trailer) within Hu...",5,10,32 weeks (two full semesters),32.0,20.0,19%,4.0,23%,4.0,8.0,15.0,6.0,0.0,1.0,Neither,NaN,"standard, non-electric",University shuttle,Safety,Arcata,95521,3638 Baldwin St,LK Wood Blvd & St Louis Rd #737,knew_and_used,knew_did_not_use,knew_did_not_use,knew_did_not_use,knew_did_not_use,did_not_know_might_use,NaN,0.62,walk,-0.0272
6881,S_02498,2025-26,Staff/Admin,agree,"Off-campus residence that is not University housing (house, apartment, condo, traile

... (77 outputs left out)

,data_type
respondent_id,str
year,str
role,str
consent,str
residence_type,str
days_commuting,str
trips_per_week,str
weeks_on_campus_cat,str
weeks_per_year,float64
pct_walk_oncampus,float64



Respondent ID formatting:
Respondent IDs not following the expected S_#### pattern:


,respondent_id,year
520,F_01257,2020-21
521,F_01257,2020-21
522,F_01257,2020-21
523,F_01257,2020-21
524,F_01257,2020-21
...,...,...
7099,F_02388,2025-26
7111,F_00921,2025-26
7112,F_01417,2020-21
7114,F_00409,2023-24



DATA-QUALITY AUDIT COMPLETE


# Clean Role

In [25]:

# 2.1 STANDARDIZE ROLE VALUES


print("=" * 80)
print("STANDARDIZING ROLE VALUES")
print("=" * 80)

# Preserve original role values
if "role_original" not in com.columns:
    com["role_original"] = com["role"].copy()

# Normalize capitalization and whitespace
com["role_clean"] = (
    com["role"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
)

# Standardize role categories
role_mapping = {
    "student": "Student",
    "studnet": "Student",

    "faculty": "Faculty",

    "staff": "Staff/Admin",
    "staff/admin": "Staff/Admin",
    "staff-admin": "Staff/Admin"
}

# Apply mapping while preserving unexpected values
com["role_clean"] = (
    com["role_clean"]
    .map(role_mapping)
    .fillna(
        com["role"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
)


# COMPARE ORIGINAL AND STANDARDIZED VALUES


print("\nOriginal vs. standardized role values:")

role_comparison = (
    com.groupby(
        ["role_original", "role_clean"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(["role_clean", "role_original"])
)

display(role_comparison)


# STANDARDIZED ROLE FREQUENCIES


print("\nStandardized role frequencies:")

role_summary = (
    com["role_clean"]
    .value_counts(dropna=False)
    .rename_axis("role")
    .reset_index(name="count")
)

display(role_summary)


# IDENTIFY UNEXPECTED ROLE VALUES


print("\nUnexpected role values:")

approved_roles = [
    "Student",
    "Faculty",
    "Staff/Admin"
]

unexpected_roles = com.loc[
    com["role_clean"].notna()
    & ~com["role_clean"].isin(approved_roles),
    ["respondent_id", "role_original", "role_clean"]
]

if unexpected_roles.empty:
    print("PASS: All role values match approved categories.")
else:
    print("WARNING: Unexpected role values require review.")
    display(unexpected_roles)


# FINALIZE ROLE STANDARDIZATION


com["role"] = com["role_clean"]

com.drop(columns=["role_clean"], inplace=True)

print("\nRole standardization applied to com['role'].")

print("\nFinal role frequencies:")
display(com["role"].value_counts(dropna=False).to_frame("count"))

print("=" * 80)


STANDARDIZING ROLE VALUES

Original vs. standardized role values:


,role_original,role_clean,count
0,FACULTY,Faculty,67
1,Faculty,Faculty,1377
2,Faculty,Faculty,62
4,Staff,Staff/Admin,77
5,Staff/Admin,Staff/Admin,1518
7,staff/admin,Staff/Admin,68
3,STUDENT,Student,59
6,Student,Student,3836
8,student,Student,65



Standardized role frequencies:


,role,count
0,Student,3960
1,Staff/Admin,1663
2,Faculty,1506



Unexpected role values:
PASS: All role values match approved categories.

Role standardization applied to com['role'].

Final role frequencies:


,count
role,
Student,3960
Staff/Admin,1663
Faculty,1506


# Clean Address

In [26]:
# 5.1 STANDARDIZE CITY VALUES


print("=" * 80)
print("STANDARDIZING CITY VALUES")
print("=" * 80)

# Preserve original city values
if "city_original" not in com.columns:
    com["city_original"] = com["city"].copy()

# Normalize capitalization and whitespace
com["city_clean"] = (
    com["city"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
)

# Standardize city names
city_mapping = {
    "arcata": "Arcata",
    "arcatra": "Arcata",

    "eureka": "Eureka",
    "eurika": "Eureka",

    "mckinleyville": "McKinleyville",
    "mckinelyville": "McKinleyville",

    "fortuna": "Fortuna",
    "trinidad": "Trinidad",
    "blue lake": "Blue Lake"
}

# Apply mapping while preserving unexpected values
com["city_clean"] = (
    com["city_clean"]
    .map(city_mapping)
    .fillna(
        com["city"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
)


# COMPARE ORIGINAL AND STANDARDIZED VALUES


print("\nOriginal vs. standardized city values:")

city_comparison = (
    com.groupby(
        ["city_original", "city_clean"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values(["city_clean", "city_original"])
)

display(city_comparison)


# STANDARDIZED CITY FREQUENCIES


print("\nStandardized city frequencies:")

city_summary = (
    com["city_clean"]
    .value_counts(dropna=False)
    .rename_axis("city")
    .reset_index(name="count")
)

display(city_summary)


# IDENTIFY UNEXPECTED CITY VALUES


print("\nUnexpected city values:")

approved_cities = [
    "Arcata",
    "Eureka",
    "McKinleyville",
    "Fortuna",
    "Trinidad",
    "Blue Lake"
]

unexpected_cities = com.loc[
    com["city_clean"].notna()
    & ~com["city_clean"].isin(approved_cities),
    ["respondent_id", "city_original", "city_clean"]
]

if unexpected_cities.empty:
    print("PASS: All city values match approved categories.")
else:
    print("WARNING: Unexpected city values require review.")
    display(unexpected_cities)


# FINALIZE CITY STANDARDIZATION


com["city"] = com["city_clean"]

com.drop(columns=["city_clean"], inplace=True)

print("\nCity standardization applied to com['city'].")

print("\nFinal city frequencies:")
display(com["city"].value_counts(dropna=False).to_frame("count"))

print("=" * 80)


STANDARDIZING CITY VALUES

Original vs. standardized city values:


,city_original,city_clean,count
0,Arcata,Arcata,2647
1,Arcatra,Arcata,164
2,BLUE LAKE,Blue Lake,4
3,Blue Lake,Blue Lake,149
4,Eureka,Eureka,1983
5,Eurika,Eureka,115
6,FORTUNA,Fortuna,14
7,Fortuna,Fortuna,256
8,McKinelyville,McKinleyville,80
9,McKinleyville,McKinleyville,1520



Standardized city frequencies:


,city,count
0,Arcata,2811
1,Eureka,2098
2,McKinleyville,1600
3,Fortuna,270
4,Trinidad,197
5,Blue Lake,153



Unexpected city values:
PASS: All city values match approved categories.

City standardization applied to com['city'].

Final city frequencies:


,count
city,
Arcata,2811
Eureka,2098
McKinleyville,1600
Fortuna,270
Trinidad,197
Blue Lake,153


# Clean Transportation

In [27]:
# 7. PRIMARY TRANSPORTATION MODE STANDARDIZATION
print("=" * 80)
print("PRIMARY TRANSPORTATION MODE STANDARDIZATION")
print("=" * 80)

# Preserve original values for auditing
if "mode_primary_original" not in com.columns:
    com["mode_primary_original"] = com["mode_primary"].copy()

# Normalize whitespace and capitalization for mapping
mode_normalized = (
    com["mode_primary"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
)

# Explicit standardization mapping
mode_mapping = {
    "drive": "Drive",
    "drive alone": "Drive",

    "walk": "Walk",
    "walking": "Walk",

    "tele": "Tele",
    "bus": "Bus",

    "flying": "Flying",
    "teleport": "Teleport"
}

# Apply mapping while preserving missing and unexpected values
com["mode_primary_clean"] = mode_normalized.map(mode_mapping)

# Preserve unexpected nonmissing values for manual review
unmapped_mask = (
    mode_normalized.notna()
    & com["mode_primary_clean"].isna()
)

com.loc[unmapped_mask, "mode_primary_clean"] = (
    com.loc[unmapped_mask, "mode_primary"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)


# STANDARDIZATION COMPARISON


print("\nSTANDARDIZATION COMPARISON")
print("-" * 80)

mode_comparison = (
    com.groupby(
        ["mode_primary_original", "mode_primary_clean"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(mode_comparison)


# STANDARDIZED FREQUENCIES


print("\nSTANDARDIZED PRIMARY MODE FREQUENCIES")
print("-" * 80)

display(
    com["mode_primary_clean"]
    .value_counts(dropna=False)
    .rename_axis("mode_primary")
    .to_frame("count")
)


# UNMAPPED VALUES


print("\nUNMAPPED VALUES REQUIRING REVIEW")
print("-" * 80)

if unmapped_mask.any():
    display(
        com.loc[unmapped_mask, "mode_primary"]
        .value_counts(dropna=False)
        .rename_axis("original_value")
        .to_frame("count")
    )
else:
    print("No unexpected nonmissing values found.")


# MISSING VALUE CHECK


print("\nMISSING VALUE CHECK")
print("-" * 80)

print("Original missing values:", com["mode_primary_original"].isna().sum())
print("Standardized missing values:", com["mode_primary_clean"].isna().sum())


# FINALIZE STANDARDIZATION


com["mode_primary"] = com["mode_primary_clean"]

com.drop(columns=["mode_primary_clean"], inplace=True)

print("\nFINAL PRIMARY MODE FREQUENCIES")
print("-" * 80)

display(
    com["mode_primary"]
    .value_counts(dropna=False)
    .rename_axis("mode_primary")
    .to_frame("count")
)

print("\nPrimary transportation mode standardization applied to com.")
print("=" * 80)

PRIMARY TRANSPORTATION MODE STANDARDIZATION

STANDARDIZATION COMPARISON
--------------------------------------------------------------------------------


,mode_primary_original,mode_primary_clean,count
11,drive,Drive,2880
15,walk,Walk,1781
13,tele,Tele,703
17,NaN,NaN,386
16,walking,Walk,374
4,Drive Alone,Drive,364
2,DRIVE,Drive,192
8,WALK,Walk,161
3,Drive,Drive,57
10,bus,Bus,54



STANDARDIZED PRIMARY MODE FREQUENCIES
--------------------------------------------------------------------------------


,count
mode_primary,
Drive,3493
Walk,2316
Tele,753
NaN,386
Bus,70
Flying,58
Teleport,53



UNMAPPED VALUES REQUIRING REVIEW
--------------------------------------------------------------------------------
No unexpected nonmissing values found.

MISSING VALUE CHECK
--------------------------------------------------------------------------------
Original missing values: 386
Standardized missing values: 386

FINAL PRIMARY MODE FREQUENCIES
--------------------------------------------------------------------------------


,count
mode_primary,
Drive,3493
Walk,2316
Tele,753
NaN,386
Bus,70
Flying,58
Teleport,53



Primary transportation mode standardization applied to com.


# Clean Resident Housing

In [28]:
# 6.6 RESIDENCE TYPE STANDARDIZATION


print("=" * 80)
print("STANDARDIZING RESIDENCE TYPE CATEGORIES")
print("=" * 80)

# Preserve original values
com["residence_type_original"] = com["residence_type"]

# Normalize values for matching
com["residence_type_key"] = (
    com["residence_type"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
)

# Standardize residence categories
residence_mapping = {

    # Off-campus non-university housing
    "off-campus residence that is not university housing (house, apartment, condo, trailer) within humboldt county":
        "Off-campus residence that is not university housing (house, apartment, condo, trailer) within Humboldt County",

    "off campus humboldt":
        "Off-campus residence that is not university housing (house, apartment, condo, trailer) within Humboldt County",

    # On-campus university housing
    "on-campus university housing (college creek, creekview, cypress, canyon, redwood, sunset)":
        "On-campus university housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)",

    "on-campus housing":
        "On-campus university housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)",

    "on campus":
        "On-campus university housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)",

    # Off-campus university housing
    "off-campus university housing (hinarr hu moulik)":
        "Off-campus university housing (Hinarr Hu Moulik)",

    "hinarr hu moulik":
        "Off-campus university housing (Hinarr Hu Moulik)",

    # Housing insecurity
    "housing insecure with no consistent place to sleep (homeless, couch-surfing, sheltering in vehicle, living in homeless shelter)":
        "Housing insecure with no consistent place to sleep (homeless, couch-surfing, sheltering in vehicle, living in homeless shelter)"
}

# Apply mapping
com["residence_type_clean"] = (
    com["residence_type_key"]
    .map(residence_mapping)
)


# 6.7 ORIGINAL VS. STANDARDIZED RESIDENCE CATEGORIES


print("\nOriginal vs. standardized residence categories:")

residence_comparison = (
    com.groupby(
        ["residence_type_original", "residence_type_clean"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("residence_type_clean")
)

display(residence_comparison)

print("\nStandardized residence frequencies:")

display(
    com["residence_type_clean"]
    .value_counts(dropna=False)
)


# 6.8 RESIDENCE TYPE VALIDATION


print("=" * 80)
print("RESIDENCE TYPE STANDARDIZATION VALIDATION")
print("=" * 80)

# Identify unmapped values
unmapped_residences = com.loc[
    com["residence_type"].notna() &
    com["residence_type_clean"].isna(),
    "residence_type_original"
].value_counts()

print("\nUnmapped residence categories:")
display(unmapped_residences)

print("\nTotal records:", len(com))
print("Original missing values:", com["residence_type"].isna().sum())
print("Standardized missing values:", com["residence_type_clean"].isna().sum())

print("\nFinal category frequencies:")
display(com["residence_type_clean"].value_counts(dropna=False))


# 6.9 FINALIZE RESIDENCE TYPE STANDARDIZATION


# Replace original column
com["residence_type"] = com["residence_type_clean"]

# Remove temporary audit columns
com.drop(
    columns=[
        "residence_type_original",
        "residence_type_key",
        "residence_type_clean"
    ],
    inplace=True
)

print("=" * 80)
print("RESIDENCE TYPE STANDARDIZATION COMPLETE")
print("=" * 80)

display(com["residence_type"].value_counts(dropna=False))

STANDARDIZING RESIDENCE TYPE CATEGORIES

Original vs. standardized residence categories:


,residence_type_original,residence_type_clean,count
1,"Housing insecure with no consistent place to sleep (homeless, couch-surfing, sheltering in vehic...","Housing insecure with no consistent place to sleep (homeless, couch-surfing, sheltering in vehic...",56
2,"OFF-CAMPUS RESIDENCE THAT IS NOT UNIVERSITY HOUSING (HOUSE, APARTMENT, CONDO, TRAILER) WITHIN HU...","Off-campus residence that is not university housing (house, apartment, condo, trailer) within Hu...",113
4,"Off-campus residence that is not University housing (house, apartment, condo, trailer) within Hu...","Off-campus residence that is not university housing (house, apartment, condo, trailer) within Hu...",6399
7,off campus humboldt,"Off-campus residence that is not university housing (house, apartment, condo, trailer) within Hu...",116
0,Hinarr Hu Moulik,Off-campus university housing (Hinarr Hu Moulik),2
3,Off-campus University housing (Hinarr Hu Moulik),Off-campus university housing (Hinarr Hu Moulik),73
5,On-Campus Housing,"On-campus university housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)",9
6,"On-campus University housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)","On-campus university housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)",358
8,on campus,"On-campus university housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)",3



Standardized residence frequencies:


residence_type_clean
Off-campus residence that is not university housing (house, apartment, condo, trailer) within Humboldt County                      6628
On-campus university housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)                                           370
Off-campus university housing (Hinarr Hu Moulik)                                                                                     75
Housing insecure with no consistent place to sleep (homeless, couch-surfing, sheltering in vehicle, living in homeless shelter)      56
Name: count, dtype: int64

RESIDENCE TYPE STANDARDIZATION VALIDATION

Unmapped residence categories:


Series([], Name: count, dtype: int64)


Total records: 7129
Original missing values: 0
Standardized missing values: 0

Final category frequencies:


residence_type_clean
Off-campus residence that is not university housing (house, apartment, condo, trailer) within Humboldt County                      6628
On-campus university housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)                                           370
Off-campus university housing (Hinarr Hu Moulik)                                                                                     75
Housing insecure with no consistent place to sleep (homeless, couch-surfing, sheltering in vehicle, living in homeless shelter)      56
Name: count, dtype: int64

RESIDENCE TYPE STANDARDIZATION COMPLETE


residence_type
Off-campus residence that is not university housing (house, apartment, condo, trailer) within Humboldt County                      6628
On-campus university housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)                                           370
Off-campus university housing (Hinarr Hu Moulik)                                                                                     75
Housing insecure with no consistent place to sleep (homeless, couch-surfing, sheltering in vehicle, living in homeless shelter)      56
Name: count, dtype: int64

# Audit

In [29]:

print("=" * 85)
print("FINAL DATA QUALITY AUDIT — COM DATASET")
print("=" * 85)

# DEFINE EXPECTED DATASET STRUCTURE


expected_columns = [
    "respondent_id",
    "year",
    "role",
    "consent",
    "residence_type",
    "days_commuting",
    "trips_per_week",
    "weeks_on_campus_cat",
    "weeks_per_year",
    "pct_walk_oncampus",
    "pct_walk_offcampus",
    "pct_tele",
    "pct_drive",
    "pct_shuttle",
    "pct_carpool",
    "pct_bus",
    "pct_bike",
    "pct_rideshare",
    "pct_other",
    "vehicle_type",
    "ev_charge_offcampus",
    "bike_power",
    "secondary_mode",
    "secondary_barrier",
    "city",
    "zip",
    "street",
    "nearest_intersection",
    "resource_jackpass",
    "resource_bikeshare",
    "resource_blc",
    "resource_ridematch",
    "resource_zipcar",
    "resource_express_bus",
    "open_ended_no_car",
    "distance_to_campus_miles",
    "mode_primary",
    "mtcde_est"
]

missing_columns = [
    column for column in expected_columns
    if column not in com.columns
]

unexpected_columns = [
    column for column in com.columns
    if column not in expected_columns
]

# Store results without modifying the dataset.
critical_checks = {}
warning_checks = {}


# 1. DATASET STRUCTURE


print("\n1. DATASET STRUCTURE")
print("-" * 85)

print(f"Total records: {len(com):,}")
print(f"Total columns: {len(com.columns):,}")

if missing_columns:
    print(f"FAIL: Missing expected columns: {missing_columns}")
    critical_checks["Expected columns present"] = False
else:
    print("PASS: All expected columns are present.")
    critical_checks["Expected columns present"] = True

if unexpected_columns:
    print(f"WARNING: Unexpected columns: {unexpected_columns}")
else:
    print("PASS: No unexpected columns.")


# 2. MISSING VALUE AUDIT


print("\n2. MISSING VALUE AUDIT")
print("-" * 85)

missing_summary = pd.DataFrame({
    "missing_values": com.isna().sum(),
    "missing_percentage": (com.isna().mean() * 100).round(2)
}).sort_values("missing_percentage", ascending=False)

display(missing_summary)

total_missing = int(com.isna().sum().sum())
total_cells = com.shape[0] * com.shape[1]

print(f"Total missing cells: {total_missing:,}")
print(
    f"Overall completeness: "
    f"{((1 - total_missing / total_cells) * 100):.2f}%"
    if total_cells > 0 else "Overall completeness: N/A"
)

if total_missing == 0:
    print("PASS: No missing values found.")
else:
    print("WARNING: Missing values remain.")
    print("Review whether missing responses are legitimate survey nonresponses.")

warning_checks["Missing values reviewed"] = total_missing == 0


# 3. RESPONDENT ID AUDIT


print("\n3. RESPONDENT ID AUDIT")
print("-" * 85)

if "respondent_id" in com.columns:

    missing_ids = com["respondent_id"].isna().sum()

    duplicate_ids = com.loc[
        com["respondent_id"].notna()
        & com["respondent_id"].duplicated(keep=False)
    ].sort_values("respondent_id")

    print(f"Missing respondent IDs: {missing_ids:,}")
    print(f"Rows with repeated respondent IDs: {len(duplicate_ids):,}")
    print(
        f"Unique repeated respondent IDs: "
        f"{duplicate_ids['respondent_id'].nunique():,}"
    )

    if missing_ids == 0:
        print("PASS: No missing respondent IDs.")
        critical_checks["No missing respondent IDs"] = True
    else:
        print("FAIL: Missing respondent IDs remain.")
        critical_checks["No missing respondent IDs"] = False

    if duplicate_ids.empty:
        print("PASS: No repeated respondent IDs.")
    else:
        print("WARNING: Repeated respondent IDs require review.")
        display(duplicate_ids)

    # Repeated respondent IDs are not automatically a failure.
    # Confirm the intended unit of observation before enforcing uniqueness.
    warning_checks["Respondent IDs are unique or justified"] = duplicate_ids.empty

else:
    print("FAIL: respondent_id column is missing.")
    critical_checks["No missing respondent IDs"] = False


# 4. ROLE AUDIT


print("\n4. ROLE AUDIT")
print("-" * 85)

if "role" in com.columns:

    role_mapping = {
        "student": "Student",
        "staff/admin": "Staff/Admin",
        "faculty": "Faculty"
    }

    normalized_roles = (
        com["role"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    standardized_roles = normalized_roles.map(role_mapping)

    unexpected_roles = com.loc[
        standardized_roles.isna(),
        ["respondent_id", "role"]
    ]

    role_standardization_issues = com.loc[
        standardized_roles.notna()
        & (
            com["role"].astype("string").str.strip()
            != standardized_roles
        ),
        ["respondent_id", "role"]
    ]

    print("Role frequencies:")
    display(com["role"].value_counts(dropna=False).to_frame("count"))

    if unexpected_roles.empty:
        print("PASS: All roles are recognized.")
        critical_checks["Recognized roles"] = True
    else:
        print("FAIL: Unexpected or missing roles found.")
        display(unexpected_roles)
        critical_checks["Recognized roles"] = False

    if role_standardization_issues.empty:
        print("PASS: Role labels are standardized.")
    else:
        print("WARNING: Role labels require standardization.")
        display(role_standardization_issues)

    warning_checks["Role labels standardized"] = (
        role_standardization_issues.empty
    )

else:
    print("FAIL: role column is missing.")
    critical_checks["Recognized roles"] = False


# 5. RESIDENCE TYPE AUDIT


print("\n5. RESIDENCE TYPE AUDIT")
print("-" * 85)

if "residence_type" in com.columns:

    approved_residence_types = {
        "off-campus residence that is not university housing "
        "(house, apartment, condo, trailer) within humboldt county",

        "on-campus university housing "
        "(college creek, creekview, cypress, canyon, redwood, sunset)",

        "off-campus university housing (hinarr hu moulik)",

        "housing insecure with no consistent place to sleep "
        "(homeless, couch-surfing, sheltering in vehicle, "
        "living in homeless shelter)"
    }

    normalized_residence = (
        com["residence_type"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.casefold()
    )

    unexpected_residence = com.loc[
        normalized_residence.notna()
        & ~normalized_residence.isin(approved_residence_types),
        ["respondent_id", "residence_type"]
    ]

    print("Residence type frequencies:")
    display(
        com["residence_type"]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    if unexpected_residence.empty:
        print("PASS: All populated residence types match approved categories.")
        critical_checks["Approved residence categories"] = True
    else:
        print("FAIL: Unrecognized residence categories remain.")
        display(unexpected_residence)
        critical_checks["Approved residence categories"] = False

    warning_checks["Residence type missing values"] = (
        com["residence_type"].isna().sum() == 0
    )

else:
    print("FAIL: residence_type column is missing.")
    critical_checks["Approved residence categories"] = False


# 6. CITY AUDIT


print("\n6. CITY AUDIT")
print("-" * 85)

if "city" in com.columns:

    approved_cities = {
        "arcata",
        "eureka",
        "mckinleyville",
        "fortuna",
        "trinidad",
        "blue lake"
    }

    normalized_city = (
        com["city"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    unexpected_cities = com.loc[
        normalized_city.notna()
        & ~normalized_city.isin(approved_cities),
        ["respondent_id", "city"]
    ]

    print("City frequencies:")
    display(com["city"].value_counts(dropna=False).to_frame("count"))

    if unexpected_cities.empty:
        print("PASS: All populated city values match approved cities.")
    else:
        print("WARNING: Unexpected city values require review.")
        display(unexpected_cities)

    warning_checks["City values recognized"] = unexpected_cities.empty

else:
    print("WARNING: city column is missing.")


# 7. PRIMARY TRANSPORTATION MODE AUDIT


print("\n7. PRIMARY TRANSPORTATION MODE AUDIT")
print("-" * 85)

if "mode_primary" in com.columns:

    approved_modes = {
        "bus",
        "drive",
        "flying",
        "tele",
        "teleport",
        "walk"
    }

    normalized_modes = (
        com["mode_primary"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    unexpected_modes = com.loc[
        normalized_modes.notna()
        & ~normalized_modes.isin(approved_modes),
        ["respondent_id", "mode_primary"]
    ]

    print("Primary mode frequencies:")
    display(
        com["mode_primary"]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    if unexpected_modes.empty:
        print("PASS: All populated primary modes are recognized.")
    else:
        print("WARNING: Unrecognized primary mode values remain.")
        display(unexpected_modes)

    warning_checks["Primary modes recognized"] = unexpected_modes.empty

else:
    print("WARNING: mode_primary column is missing.")


# 8. YEAR FORMAT AUDIT


print("\n8. YEAR FORMAT AUDIT")
print("-" * 85)

if "year" in com.columns:

    normalized_year = (
        com["year"]
        .astype("string")
        .str.strip()
    )

    year_format_issues = com.loc[
        normalized_year.notna()
        & ~normalized_year.str.fullmatch(
            r"\d{4}-\d{2}", na=False
        ),
        ["respondent_id", "year"]
    ]

    print("Year frequencies:")
    display(com["year"].value_counts(dropna=False).to_frame("count"))

    if year_format_issues.empty:
        print("PASS: All populated year values follow YYYY-YY format.")
    else:
        print("WARNING: Year formats require review.")
        display(year_format_issues)

    warning_checks["Year formats standardized"] = year_format_issues.empty

else:
    print("WARNING: year column is missing.")


# 9. PERCENTAGE VARIABLE AUDIT


print("\n9. PERCENTAGE VARIABLE AUDIT")
print("-" * 85)

percentage_columns = [
    "pct_walk_oncampus",
    "pct_walk_offcampus",
    "pct_tele",
    "pct_drive",
    "pct_shuttle",
    "pct_carpool",
    "pct_bus",
    "pct_bike",
    "pct_rideshare",
    "pct_other"
]

available_percentage_columns = [
    column for column in percentage_columns
    if column in com.columns
]

percentage_issues = []
percentage_sentinels = []
percentage_non_numeric = []

for column in available_percentage_columns:

    numeric_values = pd.to_numeric(com[column], errors="coerce")

    non_numeric_mask = com[column].notna() & numeric_values.isna()

    negative_mask = numeric_values.notna() & (numeric_values < 0)
    above_100_mask = numeric_values.notna() & (numeric_values > 100)

    for index in com.index[non_numeric_mask]:
        percentage_non_numeric.append({
            "respondent_id": com.loc[index, "respondent_id"],
            "column": column,
            "value": com.loc[index, column]
        })

    for index in com.index[negative_mask]:
        value = numeric_values.loc[index]

        record = {
            "respondent_id": com.loc[index, "respondent_id"],
            "column": column,
            "value": value
        }

        # Known special codes previously identified in pct_other.
        if column == "pct_other" and value in [-1, -2, -3]:
            percentage_sentinels.append(record)
        else:
            percentage_issues.append(record)

    for index in com.index[above_100_mask]:
        percentage_issues.append({
            "respondent_id": com.loc[index, "respondent_id"],
            "column": column,
            "value": numeric_values.loc[index]
        })

percentage_issues = pd.DataFrame(percentage_issues)
percentage_sentinels = pd.DataFrame(percentage_sentinels)
percentage_non_numeric = pd.DataFrame(percentage_non_numeric)

print(f"Percentage columns audited: {len(available_percentage_columns)}")

if percentage_non_numeric.empty:
    print("PASS: No non-numeric percentage values found.")
else:
    print("FAIL: Non-numeric values found in percentage columns.")
    display(percentage_non_numeric)

if percentage_issues.empty:
    print("PASS: No unexpected negative values or values above 100.")
else:
    print("FAIL: Invalid percentage values found.")
    display(percentage_issues)

if percentage_sentinels.empty:
    print("PASS: No known sentinel codes found in pct_other.")
else:
    print("WARNING: Known sentinel codes remain in pct_other.")
    display(percentage_sentinels)

critical_checks["Numeric percentage values"] = (
    percentage_non_numeric.empty and percentage_issues.empty
)

warning_checks["Known percentage sentinel codes reviewed"] = (
    percentage_sentinels.empty
)


# 10. OTHER NUMERIC VARIABLE AUDIT


print("\n10. NUMERIC VARIABLE AUDIT")
print("-" * 85)

numeric_columns = [
    "days_commuting",
    "trips_per_week",
    "weeks_per_year",
    "distance_to_campus_miles",
    "mtcde_est"
]

available_numeric_columns = [
    column for column in numeric_columns
    if column in com.columns
]

numeric_conversion_issues = []

for column in available_numeric_columns:

    numeric_values = pd.to_numeric(com[column], errors="coerce")

    invalid_mask = com[column].notna() & numeric_values.isna()

    for index in com.index[invalid_mask]:
        numeric_conversion_issues.append({
            "respondent_id": com.loc[index, "respondent_id"],
            "column": column,
            "value": com.loc[index, column]
        })

numeric_conversion_issues = pd.DataFrame(numeric_conversion_issues)

if numeric_conversion_issues.empty:
    print("PASS: All populated numeric fields contain numeric values.")
else:
    print("FAIL: Non-numeric values found in numeric fields.")
    display(numeric_conversion_issues)

critical_checks["Numeric field validity"] = (
    numeric_conversion_issues.empty
)

# Review negative values separately.
negative_numeric_values = []

for column in available_numeric_columns:

    numeric_values = pd.to_numeric(com[column], errors="coerce")

    negative_mask = numeric_values.notna() & (numeric_values < 0)

    for index in com.index[negative_mask]:
        negative_numeric_values.append({
            "respondent_id": com.loc[index, "respondent_id"],
            "column": column,
            "value": numeric_values.loc[index]
        })

negative_numeric_values = pd.DataFrame(negative_numeric_values)

if negative_numeric_values.empty:
    print("PASS: No negative values found in audited numeric fields.")
else:
    print("WARNING: Negative numeric values require review.")
    display(negative_numeric_values)

warning_checks["Negative numeric values reviewed"] = (
    negative_numeric_values.empty
)


# 11. ZIP CODE FORMAT AUDIT


print("\n11. ZIP CODE FORMAT AUDIT")
print("-" * 85)

if "zip" in com.columns:

    normalized_zip = (
        com["zip"]
        .astype("string")
        .str.strip()
    )

    invalid_zip_mask = (
        normalized_zip.notna()
        & ~normalized_zip.str.fullmatch(r"\d{5}", na=False)
    )

    zip_issues = com.loc[
        invalid_zip_mask,
        ["respondent_id", "city", "zip"]
    ]

    if zip_issues.empty:
        print("PASS: All populated ZIP codes contain five digits.")
    else:
        print("WARNING: Invalid ZIP formats found.")
        display(zip_issues)

    warning_checks["ZIP formats valid"] = zip_issues.empty

else:
    print("WARNING: zip column is missing.")


# 12. EXACT DUPLICATE RECORD AUDIT


print("\n12. EXACT DUPLICATE RECORD AUDIT")
print("-" * 85)

exact_duplicate_mask = com.duplicated(keep=False)

exact_duplicates = com.loc[
    exact_duplicate_mask
].sort_values(
    "respondent_id"
    if "respondent_id" in com.columns
    else com.columns[0]
)

print(f"Rows involved in exact duplicates: {len(exact_duplicates):,}")

if exact_duplicates.empty:
    print("PASS: No completely identical records found.")
else:
    print("WARNING: Completely identical records found.")
    display(exact_duplicates)

warning_checks["No exact duplicate records"] = exact_duplicates.empty


# 13. CATEGORICAL COMPLETENESS AND CONSISTENCY


print("\n13. CATEGORICAL VALUE REVIEW")
print("-" * 85)

categorical_columns = [
    "consent",
    "weeks_on_campus_cat",
    "vehicle_type",
    "ev_charge_offcampus",
    "bike_power",
    "secondary_mode",
    "secondary_barrier",
    "resource_jackpass",
    "resource_bikeshare",
    "resource_blc",
    "resource_ridematch",
    "resource_zipcar",
    "resource_express_bus"
]

for column in categorical_columns:

    if column in com.columns:

        print(f"\n{column}")
        print("-" * 50)

        display(
            com[column]
            .value_counts(dropna=False)
            .to_frame("count")
        )


# 14. FINAL DATASET READINESS


print("\n" + "=" * 85)
print("FINAL DATASET READINESS")
print("=" * 85)

print("\nCRITICAL VALIDATION CHECKS")
print("-" * 85)

for check, passed in critical_checks.items():

    status = "PASS" if passed else "FAIL"
    print(f"{status}: {check}")

print("\nDATA QUALITY WARNINGS")
print("-" * 85)

for check, passed in warning_checks.items():

    status = "PASS" if passed else "WARNING"
    print(f"{status}: {check}")

all_critical_checks_passed = all(critical_checks.values())

print("\n" + "=" * 85)

if all_critical_checks_passed:

    print("RESULT: PASS — DATASET PASSES ALL CRITICAL VALIDATION CHECKS.")

    if not all(warning_checks.values()):
        print(
            "NOTE: Non-critical data quality warnings remain. "
            "Review them before final export."
        )
    else:
        print("RESULT: No outstanding audit warnings.")

else:

    print("RESULT: FAIL — DATASET DOES NOT PASS ALL CRITICAL CHECKS.")
    print("Resolve the failed validation checks before final export.")

print("=" * 85)


FINAL DATA QUALITY AUDIT — COM DATASET

1. DATASET STRUCTURE
-------------------------------------------------------------------------------------
Total records: 7,129
Total columns: 41
PASS: All expected columns are present.

2. MISSING VALUE AUDIT
-------------------------------------------------------------------------------------


,missing_values,missing_percentage
ev_charge_offcampus,6319,88.64
open_ended_no_car,4015,56.32
secondary_barrier,722,10.13
mode_primary,386,5.41
mode_primary_original,386,5.41
nearest_intersection,201,2.82
trips_per_week,168,2.36
bike_power,91,1.28
mtcde_est,89,1.25
pct_other,86,1.21


Total missing cells: 12,640
Overall completeness: 95.68%
Review whether missing responses are legitimate survey nonresponses.

3. RESPONDENT ID AUDIT
-------------------------------------------------------------------------------------
Missing respondent IDs: 0
Rows with repeated respondent IDs: 6,029
Unique repeated respondent IDs: 1,593
PASS: No missing respondent IDs.


,respondent_id,year,role,consent,residence_type,days_commuting,trips_per_week,weeks_on_campus_cat,weeks_per_year,pct_walk_oncampus,pct_walk_offcampus,pct_tele,pct_drive,pct_shuttle,pct_carpool,pct_bus,pct_bike,pct_rideshare,pct_other,vehicle_type,ev_charge_offcampus,bike_power,secondary_mode,secondary_barrier,city,zip,street,nearest_intersection,resource_jackpass,resource_bikeshare,resource_blc,resource_ridematch,resource_zipcar,resource_express_bus,open_ended_no_car,distance_to_campus_miles,mode_primary,mtcde_est,role_original,city_original,mode_primary_original
2994,F_00000,2022-23,Faculty,agree,"Off-campus residence that is not university housing (house, apartment, condo, trailer) within Hu...",6,12,44 weeks (two full semesters + one 8-week summer session),44.0,7.0,8,13.0,49,2.0,4.0,1.0,16.0,0.0,0.0,Motorcycle or moped,NaN,"standard, non-electric",Motorcycle/moped,Convenience,McKinleyville,95519,1461 Central Ave,Bates Rd & Winchester Ave,"I knew about it, but I didn't use it.","I knew about it, but I didn't use it.","I knew about it, but I didn't use it.",knew_did_not_use,"I knew about it, but I didn't use it.",knew_did_not_use,Weather makes biking hard in winter.,4.68,Drive,0.5153,Faculty,McKinleyville,drive
5382,F_00000,2024-25,Faculty,agree,"Off-campus residence that is not university housing (house, apartment, condo, trailer) within Hu...",5,10,"18 weeks (Faculty ""Green"" days for one full semester)",18.0,2.0,20,7.0,51,2.0,5.0,2.0,10.0,0.0,1.0,Neither,NaN,"standard, non-electric",Public transit,Convenience,McKinleyville,95519,5064 Hiller Rd,Central Ave & School Rd,knew_did_not_use,knew_did_not_use,did_not_know_might_use,knew_did_not_use,knew_did_not_use,did_not_know_wouldnt_use,More frequent buses aligned with class times.,4.87,Drive,0.2091,Faculty,McKinleyville,drive
6572,F_00004,2025-26,Faculty,agree,"On-campus university housing (College Creek, Creekview, Cypress, Canyon, Redwood, Sunset)",7,14,"18 weeks (Faculty ""Green"" days for one full semester)",18.0,5.0,14,12.0,0,1.0,0.0,0.0,12.0,0.0,0.0,Plug-in electric vehicle,91-100%,electric,University shuttle,NaN,Arcata,95521,1221 LK Wood Blvd,St Louis Rd #459 & St Louis Rd #611,knew_did_not_use,knew_did_not_use,did_not_know_wouldnt_use,knew_did_not_use,knew_did_not_use,did_not_know_might_use,NaN,0.48,Drive,0.0606,Faculty,Arcata,drive
6684,F_00004,2025-26,Faculty,agree,"Off-campus residence that is not university housing (house, apartment, condo, trailer) within Hu...",0,3,about a year,44.0,5.0,13,13.0,43,1.0,4.0,2.0,19.0,0.0,0.0,Plug-in electric vehicle,20-29%,electric,University shuttle,Flexibility,Arcata,95521,6442 G St,St Louis Rd #351 & St Louis Rd #519,did_not_know_wouldnt_use,knew_did_not_use,did_not_know_might_use,did_not_know_wouldnt_use,did_not_know_wouldnt_use,knew_did_not_use,More reliable evening transit.,0.61,Drive,0.0448,Faculty,Arcata,DRIVE
4153,F_00005,2023-24,Faculty,agree,"Off-campus residence that is not university housing (house, apartment, condo, trailer) within Hu...",5,10,"<18 weeks (less than Faculty ""Green"" days for one full semester)",14.0,4.0,17,12.0,47,0.0,5.0,5.0,10.0,0.0,0.0,Plug-in electric vehicle,40-49%,"standard, non-electric",Carpool/ride with others,Comfort,McKinleyville,95519,5516 Murray Rd,Bates Rd & School Rd,knew_and_used,knew_did_not_use,knew_did_not_use,did_not_know_might_use,did_not_know_might_use,knew_did_not_use,NaN,5.85,Drive,0.174,Faculty,McKinleyville,drive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6028,S_02498,2025-26,Student,agree,"Off-campus residence that is not university housing (house, apartment, condo, trailer) within Hu...",5,10,32 weeks (two full semesters),32.0,20.0,19%,4.0,23%,4.0,8.0,15.0,6.0,0.0,1.0,Neither,NaN,"standard, non-electric",University shuttle,Safety,Arcata,95521,3638 Baldwin St,LK Wood Blvd & St Louis Rd #737,knew_and_used,knew_did_not_use,knew_did_not_use,knew_did_not_us


4. ROLE AUDIT
-------------------------------------------------------------------------------------
Role frequencies:


,count
role,
Student,3960
Staff/Admin,1663
Faculty,1506


... (44 outputs left out)

,count
resource_zipcar,
knew_did_not_use,4715
did_not_know_wouldnt_use,1005
did_not_know_might_use,844
knew_and_used,251
"I knew about it, but I didn't use it.",212
"I did not know about it, but I probably wouldn't have used it anyway.",40
"I didn't know about it, but now that I do I'll probably use it!",27
NaN,14
"I knew about it, and I used it!",12



resource_express_bus
--------------------------------------------------


,count
resource_express_bus,
knew_did_not_use,3150
did_not_know_wouldnt_use,1797
did_not_know_might_use,1439
knew_and_used,442
"I knew about it, but I didn't use it.",134
"I did not know about it, but I probably wouldn't have used it anyway.",68
"I didn't know about it, but now that I do I'll probably use it!",62
"I knew about it, and I used it!",20
NaN,13



FINAL DATASET READINESS

CRITICAL VALIDATION CHECKS
-------------------------------------------------------------------------------------
PASS: Expected columns present
PASS: No missing respondent IDs
PASS: Recognized roles
PASS: Approved residence categories
FAIL: Numeric percentage values
FAIL: Numeric field validity

DATA QUALITY WARNINGS
-------------------------------------------------------------------------------------
PASS: Role labels standardized
PASS: Residence type missing values
PASS: City values recognized
PASS: Primary modes recognized

RESULT: FAIL — DATASET DOES NOT PASS ALL CRITICAL CHECKS.
Resolve the failed validation checks before final export.


# Save "New Dataset"

In [30]:
# Define the output directory
output_dir = Path(
    r"C:\Users\MARE\Documents\data311_project2\data\processed"
)

# Create the directory if it does not exist
output_dir.mkdir(parents=True, exist_ok=True)

# Define the output filename
output_file = output_dir / "commuter_survey_v1.csv"

# Save the cleaned DataFrame
com.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully.")
print("File location:", output_file)
print("Number of records saved:", len(com))

Cleaned dataset saved successfully.
File location: C:\Users\MARE\Documents\data311_project2\data\processed\commuter_survey_v1.csv
Number of records saved: 7129
